# D5 Abstract-Retrieve-Refine — Paper-Readiness Audit

**Generated**: 2026-04-28 by Claude (paper-readiness audit, fully grounded in source code + raw run data — markdown summaries cross-checked, not trusted on their own).

**Scope**: every D5-relevant artifact between 2026-04-23 (project kickoff) and 2026-04-29 (latest run). 14 trainer scripts, ~30 evaluation/audit/pairwise scripts, 14 data-pipeline scripts, ~100 run directories, 16 knowledge documents, 14 findings, 8 methodology docs, 2 abstract drafts.

**Reading order**: read §1–§3 in order (timeline + critical caveats). §4–§7 are detail/appendix material.

---


## 1. Executive summary

D5 trains a 30B language model (Qwen3-30B-A3B) to produce **long-form research plans** by combining (a) an Opus-extracted "oracle" of methodological items from a paper's bibliography, (b) plan-level critique from an Opus reviewer that has the source paper as privileged information, and (c) self-distillation training so that the un-reviewed model approaches its reviewed-conditioned behaviour.

**Single-goal pilot**: TTT-Discover (arxiv 2601.16175). One paper, one bibliography, one reference plan. Cross-goal validation done on 3 forward-citation follow-ups (Meta-TTL, TT-Control, Tool-Verification-TTRL) and 4 cross-domain papers (climate, bio, econ, chem) catalogued for future Phase 4b work.

**Headline result (Phase 2)**: μ-v4 iter-4 checkpoint, lr=5e-5, lifts the model from σ baseline 25.25/45 → 28.00/45 (+2.75 absolute on 9-dim Opus rubric, n=8 plans). Pairwise corroboration: 20/24 wins over {σ, δ, α} (6-2, 7-1, 7-1). The first 30B-trained variant in the project to beat the σ frozen+oracle baseline.

**Hard-won negatives** (these are what makes the paper honest):
- **F3 multi-round SDPO instability**: at every learning rate tested (1e-5, 5e-5, 2e-4) the policy eventually collapses past iter 4–6. μ-v4's iter-5 cliff drops 12.4 points in one iter. Replicates Hübotter 2026 §4 on a different task.
- **F4 critique-token-blindness**: the SDPO advantage routes through *stylistic* tokens, not content tokens. PUCT/J_β formulas appear in TEACHER (PICK) rollouts but never in STUDENT (EVAL) rollouts at any iter, any lr. Iter-4's gain is real but it isn't the gain we hypothesized.
- **F7 cross-goal NULL** (n=24 stress-tested): μ-v4 LoRA does not transfer to forward-citation follow-up papers. Original 2-plan audit "Strong-mixed 2/3 pass" was an anchoring artifact.
- **F8 v1 retracted to F8 v2**: τ_v4 distillation pipeline's original 26.50 / 7-1 STRONG headline was a lucky-draw artifact; reseeded τ_v4_clean = 24.50 with 95% CI [22.96, 26.04], pairwise 4-4 TIE vs σ_v4. The result is "comparable + 3 characterized failure modes", not "wins".
- **F9 continual SDPO NULL**: 4-cell grid (anchor saturation × Adam-state × anchor_ce regularization). All four cells fail H1 forward learning. Phase 4b cross-domain stretch test formally DEPRECATED.
- **M8 audit-pairwise divergence (a real methodology contribution)**: 2-plan/subagent batching introduces systematic anchoring that inflates close-cluster Δ by ~2–3 points. Strict 1-plan/subagent + pairwise tournament are required for verdicts at Δ < 5/45. This is the most rigorous methodology finding in the project.

**Targeted venue**: ICLR 2027 main track. Deadline ~2026-10.


## 2. Critical caveats — read before writing any paper claim

These are the load-bearing facts that, if ignored, will produce false claims in the manuscript.

### 2.1 Terminology — "SDPO" in D5 trainer code is NOT canonical SDPO

The D5 code (μ-v2, μ-v3, μ-v4, μ-v6, κ-v1) is named and documented as "SDPO". Verified against the actual source code and the upstream papers (Hübotter 2026 arxiv 2601.20802, Zhao 2026 arxiv 2601.18734):

- **Hübotter SDPO** (canonical, 2601.20802): student samples on-policy + KL divergence loss between (with-review, no-review) policies.
- **Zhao OPSD** (canonical, 2601.18734): student samples on-policy + sampled-token policy gradient (or JS).
- **D5 μ-v2 through μ-v6** (this project's "SDPO"): TEACHER samples a plan under (goal+oracle+critique) context. STUDENT logprobs are recomputed under (goal+oracle) context — i.e., on the teacher's tokens, *off-policy*. Per-token advantage `A_t = clamp(scale·(teacher_lp − student_lp), [-clip, clip])` is fed into Tinker's `importance_sampling` loss against the student prefix. **This is an off-policy IS-loss variant — a D5 in-house design, not in any paper.**
- **D5 μ-v7-opd** (`opd_mode=True`): swaps the roles. STUDENT samples on-policy under (goal+oracle); TEACHER logprobs are recomputed under (goal+oracle+critique). Same advantage formula, but on-manifold gradients. Closest to OPSD's sampled-token variant (Zhao Table 3 records ~2% gap to canonical).

**Action item for paper**: do *not* claim canonical SDPO. Either describe the D5 variant honestly ("we use an off-policy importance-sampling distillation loss with a privileged-information critic") or reframe around μ-v7-opd's on-policy variant. Both are defensible; the wrong move is conflating them.

Source documentation of this caveat: `knowledge/current/CANONICAL_NAMING_REFERENCE.md` (verified 2026-04-27 PM via Hübotter+Zhao paper read).

### 2.2 STATUS.md / DECISIONS.md numerical claims — what was verified vs not

This audit cross-checked the headline numbers in `STATUS.md` against raw `runs/*/audit_responses/*.json`. Verdict:

| Claim | Status |
|---|---|
| σ = 25.25, δ = 25.12, α = 25.00, μ-v2 = 23.88, β = 22.75, ξ = 15.62 | ✅ all exact |
| ε = 33.62 | ✅ exact |
| μ-v4 trajectory 24.00/24.75/24.88/27.25/28.00/15.62 | ✅ raw audits match (the relevant raw files do score in /36 internally; the 8-baseline summary table uses /45 — values consistent once scale is fixed) |
| Pairwise μ-v4 vs σ/δ/α: 6-2 / 7-1 / 7-1 = 20/24 | ✅ exact |
| Phase 4a cross-goal Δ +2.12 / -0.88 / +3.00 | ✅ exact (±0.01 rounding) |
| D-series n=24 pairwise meta_ttl 14/9/1, tool_v_ttrl 12/10/2 | ✅ exact |
| τ_v4 F8 v1 = 26.50 + 7-1 STRONG | ⚠ RETRACTED — lucky draw artifact |
| τ_v4_clean F8 v2 = 24.50 (CI [22.96, 26.04]) + 4-4 TIE | ✅ exact |
| Pre-realignment μ baseline = 8.88/20 | ✅ last-3-iter mean exact |
| κ-v1 smoke verdict 24.38 | ⚠ INVALID — STATUS itself marks retracted; raw iter-0 = 24.88 |
| μ-v7 OPD iter-4 = 26.625, iter-5 = 23.125 | ✅ exact |
| Phase 5 4-cell /20 best scores (8.38, 9.00, 9.375, 10.375) | ⚠ INCONCLUSIVE — audit files in those format not located in run dirs (could be in training logs not opened) |

**Conclusion**: the core Phase 2 baseline matrix and Phase 2 SDPO results are SOLID. The retracted F8 v1 + retracted Phase 4a "Strong-mixed" + retracted κ-v1 are explicitly retracted in markdown but care needed when reading older docs.

### 2.3 Plan-prompt versions matter — v3 vs v4 are different baselines

- `_KAPPA_PLAN_FOOTER` (v3): plain "600-750 words, structured sections, integrate distillations".
- `_KAPPA_PLAN_FOOTER_V4` (added 2026-04-27): adds **mandatory** Disentanglement-Ablation, Compute-Accounting (operational units), and Quantified-Frozen-LLM-Baseline-Gap directives. Plus a "citation hygiene" warning that index labels like "Methodology 17" are NOT real citations.
- σ_v4 baseline uses v4 footer with same mandatory directives in the σ context. σ baseline number 25.25 comes from a *different* prompt than σ_v4 25.75.

So Phase 2 σ (25.25, with `mu_prompts_v2`) and Phase 3 σ_v4 (25.75, with v4 footer) are NOT the same condition. Paper must label which σ is the comparator.

### 2.4 audit_v3 ISOLATED is the gold standard

Daemon audit (4-dim, Phase 0) saturates at the anchor level — F6 documents it scores all μ-v4 iters identically at 7.00/20 even when isolated audit shows 24.00 → 28.00 trajectory. audit_v3 batched (`audit_v3_runner.py`) showed within-batch templating bias. audit_v3 SHUFFLED (one big 56-plan batch) reduced bias but still anchored. audit_v3 ISOLATED (8 balanced batches × 7 plans, 1 per baseline, anonymized A–G, 1 dedicated Opus subagent per batch — defined 2026-04-26 in `audit_v3_isolated.py`) is the methodology used for all reported numbers from Phase 2F onward. This is what gives the audit its credibility and is M4's contribution.

### 2.5 "Per-instance" critic chains in κ-v1_self ≠ shared

`train_kappa_v1_self.py` runs a critique chain per instance (i ∈ 1..8) and per round (j ∈ 1..3). Each instance's chain is independent — round 2's critique sees only that instance's round-1 distillation. The cold-start critique (round 1) is a hand-written placeholder (`DISTILLATION_COLD_START_CRITIQUE`). Per-instance round-2/3 use Qwen3-30B base self-critic. Smoke A (`smoke_a_critique_correlation_v1.py`) measured Qwen self-critic vs Opus critic on 8 plans → Opus 8-0 → "self-review degeneration" failure mode confirmed.


## 3. Project timeline (chronological; one row = one decision/finding)

| Date | Phase | Event | Key number / verdict |
|------|-------|-------|---------------------|
| 2026-04-23 | 0a | Project scaffolding, plan `giggly-jumping-hollerith.md` written | Plan: 12-week schedule, ICLR 2027 target |
| 2026-04-24 | 0c | Smoke v1 (oracle abstraction → frozen 30B): A=7.00 vs B=5.12 on D3 audit | +1.88 / 20 vs goal-only |
| 2026-04-24 | 0c | Smoke v2 (with formula hints): A=11.25 vs B=5.38 | +5.875 / 20 (3× lift, 2/8 plans wrote reference's J_β formula) |
| 2026-04-24 | 0c | Smoke v3 (reasoning steps without formulas): A=9.25 vs B=6.75 | +2.50 / 20 (formula hints carry ~57% of v2 lift) |
| 2026-04-25 | 0.5 | Sanity gates: dataset migrated to D5-owned, contamination check, sampling fix (temp=1.0, top_p=0.95, no fixed seed) | 0% duplicates after fix (was 31–44%) |
| 2026-04-25 | 0.5 | Smoke v2 rerun: Δ=+4.00; v3 rerun: Δ=+3.13. Pathway claims VALID | reasoning > formula in inverted decomposition |
| 2026-04-25 | 0.6 | Phase 0.6 baseline pilot: μ baseline_v1 (10 iters, oracle v3, lr=1e-5, anchor_ce=0) | μ proxy = **8.88 / 20** (last-3-iter mean iters 4/6/8) |
| 2026-04-25 | 0.6 | Pairwise inversion discovered: μ vs δ → δ wins 8-0 (despite μ +2.13 absolute audit). δ has reference-in-context = semi-oracle | absolute audit has surface-form bias |
| 2026-04-25 | 0.6 | Fair pairwise: smoke_v3_A vs smoke_v3_B → A 8-0 (oracle helps). μ vs smoke_v3_A → A 6-2 (training adds nothing over oracle inference) | training transfer ≈ 0 in pre-realignment configuration |
| **2026-04-25 PM** | **REALIGNMENT** | **All prior runs ARCHIVED.** Issues: oracle was single-round; δ/ε mislabeled as baselines (semi-oracle); audit bias; no reviewer-standard grounding. | `runs/_archive_pre_realignment_2026_04_25/`, knowledge docs locked: `REVIEWER_STANDARDS_v1.md`, `ORACLE_DESIGN_v1.md`, `AUDIT_RUBRIC_v3.md`, `RUN_CONFIRMATION_TEMPLATE.md` |
| 2026-04-26 | 2A-bis | Bibliography rebuild from LaTeX source (replaces broken PDF v1): `extract_cite_keys → parse_bibtex → build_bibliography_v2 → expand_full_text → extract_arxiv_latex` | 90 cite keys, 86/90 resolved, 38 with full text, 28 LaTeX-→-MD converted |
| 2026-04-26 | 2A-bis | Source paper re-extracted from LaTeX (v2.md): replaces corrupted PDF v1 | source_paper/v2.md ~76,300 chars |
| 2026-04-26 | 2B | Oracle v2 build: 4-round Opus extraction over 62 papers (relevance ≥1) | final.md: 580 items in 6 categories; slim.md: ~87 items (score≥3, ≤1/category/paper); medium.md: 181 items (score≥2) |
| 2026-04-26 | 2D | Realigned baselines: α-v2, β-v2, σ-v2 (frozen+oracle), δ-v2, ε-v2, ξ-v2, μ-v2 (SDPO+oracle) | 7 baselines + μ training |
| 2026-04-26 | 2F | audit_v3 ISOLATED methodology defined: 8 balanced batches × 7 plans, anonymized plan_A..plan_G, 1 dedicated Opus subagent per batch | M4 methodology contribution |
| 2026-04-26 | 2F | 7-baseline audit_v3 ISOLATED ranking: ε=33.62 ≫ σ=25.25 ≈ δ=25.12 ≈ α=25.00 > μ-v2=23.88 > β=22.75 > ξ=15.62 | F1 finding: **σ +9.63 over ξ (oracle is biggest single effect)**; **μ-v2 LOSES to σ by -1.37 (training adds nothing at lr=1e-5)** |
| 2026-04-26 | 2 | μ-v3 launched at lr=2e-4, n_grad_steps_per_iter=4 | peak iter-0 = 24.88 → collapse iter-6 = 14.38 (mixed-script bleed, XML leaks, thinking-mode JSON leaks). User: "kill 吧" |
| 2026-04-26→27 | 2 | μ-v4 launched at lr=5e-5 (geometric median √(1e-5 × 2e-4)), eval_every=1, save_every=1, audit-drop early-stop | **iter 4 = 28.00 / 45 (+2.75 over σ)**; iter 5 cliff to 15.62 (-12.4) |
| 2026-04-27 | 2 close | mu_v4 pairwise corroboration: μ-v4 vs σ 6-2 (PRIMARY ≥6/8 PASS), vs δ 7-1, vs α 7-1 = **20/24 (83.3%)** | F2 finding paper-grade |
| 2026-04-27 | 3 design | Phase 3 LOCKED via Q1–Q10 (RETRIEVAL_DESIGN_v1.md): τ inference-time + κ training, 4 decision gates, Q9 "reuse μ-v4 recipe verbatim" | Path Y: dynamic distillation (not retrieve-then-generate) |
| 2026-04-27 | 3 | τ_v4 production single-shot: 26.50 / 7-1 STRONG vs σ_v4 (F8 v1 ORIGINAL HEADLINE) | (later retracted) |
| 2026-04-27 PM | 3 | Step 2a BoN sanity: σ_within = 1.17 (lucky draws); Step E pairwise rerun = 4-4 TIE; τ_v4_clean = 24.50 (n=8, CI [22.96, 26.04]) | **F8 v1 RETRACTED to F8 v2** "comparable + 3 failure modes" |
| 2026-04-27 | 3 | κ-v1 smoke (cold-start critic, single iter) audit = 24.88; explicitly INVALIDATED 2026-04-27 ("cost-saving compromise; SDPO advantage on cold-start text is not the same test as real critic") | κ-opus full smoke deferred |
| 2026-04-27 | 3 | Smoke A: Qwen3 self-critic vs Opus critic on 8 plans → Opus 8-0 unanimous | M7 self-review degeneration confirmed |
| 2026-04-26→27 | 4a | Phase 4a cross-goal (3 forward-citation): meta_ttl Δ +2.12, tt_control -0.88, tool_v_ttrl +3.00 → "Strong-mixed 2/3 pass" | (originally) |
| 2026-04-27 | 4a remediation | Pairwise + strict 1-plan re-audits: meta_ttl pairwise σ' 5/3 (NOT μ'); tt_control σ' edge stable; tool_v_ttrl 4/3/1 ≈ TIE | F7 DOWNGRADED to "inconclusive" |
| 2026-04-28 | 4a stress test | D-series n=24 pairwise: meta_ttl σ' 14/9/1 (58%); tool_v_ttrl 12/10/2 (50%) | **F7 DEFINITIVELY NULL** at n=24 |
| 2026-04-26 | 5 | Phase 5 4-cell continual SDPO grid: cliff_v1, cliff_v2, v3, α (anchor_ce=0.1) | All 4 fail H1: best v5_v3 Δ +0.62 (within noise); α Δ -2.62 (anchor_ce backfires) |
| 2026-04-26 | 5 | F9 finding: continual SDPO NULL across 4 cells; Phase 4b cross-domain stretch FORMALLY DEPRECATED | F9 triple-corroborated 2026-04-27 |
| 2026-04-27 | M8 | M8 contribution: 2-plan/subagent batching inflates close-cluster Δ by ~2–3 points; strict 1-plan + pairwise required for Δ < 5/45 verdicts | Standing protocol |
| 2026-04-27 | terminology | Hübotter+Zhao papers WebFetched. CANONICAL_NAMING_REFERENCE.md written | μ-v* re-labeled "D5 in-house off-policy IS-loss" |
| 2026-04-27→29 | 7 | μ-v7-opd (canonical-friendly on-policy IS-loss): iter-4 peak = 26.625 / 45 (+1.38 over σ); pairwise 8-0 over μ-v4 | F13 finding |
| 2026-04-29 | 7 | μ-v7-opd cross-goal still NULL: meta_ttl 22.875, tool_v_ttrl 23.75, tt_control 23.75; aggregate 23.458 | F14 finding (cross-goal NULL persists even with canonical algorithm) |

---


## 4. Phase-by-phase deep dive

Each phase is described with: hypothesis → method → result → paper-readiness verdict.

### 4.1 Phase 0c — smoke pathway validation (2026-04-23 → 2026-04-25 PRE-REALIGNMENT)

**Hypothesis**: "putting an Opus-extracted oracle of methodological patterns into the prompt at inference time lifts a frozen 30B's plan quality vs goal-only baseline."

**Pipeline** (`smoke_pathway_v1.py`):
- Two conditions, frozen Qwen3-30B-A3B, 8 plans each.
  - **Condition A (smoke_v3_A)**: prompt = goal + oracle abstraction (Opus-extracted patterns from one reference plan).
  - **Condition B (smoke_v3_B)**: prompt = goal only.
- Generation: temperature 1.0, top_p 0.95, max_tokens varied across smoke iterations.
- Scoring (`smoke_pathway_v1_score.py`): Qwen3-30B-A3B grader using `shared.ten_signal_reward.aggregate_reward` (4 signals — soundness, novelty, realism, rigor; /20 total).

**Three smoke iterations** (different oracle-construction strategies):
- v1 (pattern-only): A 7.00, B 5.12 → Δ +1.88 / 20.
- v2 (patterns + reasoning steps + formula hints): A 11.25, B 5.38 → Δ +5.875 (3× lift; 2/8 plans wrote reference's J_β = log Σ_a exp(β·R(a))·π(a|s) formula).
- v3 (patterns + reasoning steps, **NO formula hints**): A 9.25, B 6.75 → Δ +2.50.

**Decomposition** (after sampling-fix rerun 2026-04-25):
- Pattern structure alone: +1.88
- + Derivation reasoning steps: +0.62 marginal
- + Specific formula hints: +3.38 marginal
- Total v2 lift: +5.875

**Paper-readiness verdict**: SUPERSEDED by Phase 2F audit_v3 ISOLATED 7-baseline ranking which uses the rigorous 9-dim /45 rubric. Phase 0c smoke is historical and uses a different /20 grader. Do NOT cite Phase 0c numbers in paper. The *idea* (oracle helps frozen) survives in F1 (σ +9.63 over ξ).


### 4.2 Phase 0.5 — sanity gates (2026-04-25)

Three blockers identified from Phase 0c that needed clearing before any training:

**0.5a: Dataset migration** — D5 dataset migrated from D4 grant_proposal symlink to a self-contained copy of D3 TTT-Discover canonical files at `projects/d5_abstract_retrieve_refine/dataset/`. Files byte-verified identical. Files in scope: `research_goal.txt`, `reference_solution.txt`, `perturbations/*.txt`. D5 trainer code paths updated.

**0.5b: Qwen3 contamination check** — Qwen3-30B-A3B knowledge cutoff = March 2025. TTT-Discover paper posted 2026-01 (post-cutoff). Direct probe (3 questions about TTT-Discover concepts) confirmed no in-weights knowledge of the paper's contributions. Smoke results valid.

**0.5c: Sampling-fix verification** — Tinker `SamplingParams(seed=42)` with `num_samples=8` was producing 31–44% byte-identical outputs (deterministic-seed bug; Tinker uses the same seed for all samples in one `sample()` call). Fix: `temperature=1.0, top_p=0.95, no fixed seed`. Smoke v2/v3 reruns: 0% duplicates. v2 rerun Δ=+4.00 (vs original +5.875; 68% survival). v3 rerun Δ=+3.13 (slightly grew).

**Decomposition INVERTED after rerun**: pattern +1.88 / reasoning steps +1.25 / formula hints +0.88 (originally formula was claimed dominant at +3.38). Reasoning scaffolding is the dominant lift component once the duplicates bug is removed. Statistical caveat: n=8 too small to distinguish +4.00 vs +3.13 (gap ~1.2 SE).

**Paper-readiness verdict**: DOCUMENT ONLY (cite as motivation for the 600-750 word target and the "v4 footer" mandatory T1/T2/T3 directives). Don't cite Phase 0.5 numbers; cite Phase 2F audit_v3 ISOLATED instead.


### 4.3 Phase 0.6 — baseline pilot + pairwise discovery (2026-04-25; PRE-REALIGNMENT)

This phase is **archived** (`runs/_archive_pre_realignment_2026_04_25/`) and its numbers should NOT be cited as authoritative. It is preserved here because (a) it was the empirical motivation for the 2026-04-25 realignment, and (b) `train_mu_baseline_v1.py` is the design ancestor of train_mu_v2/v3/v4.

**Baselines run** (5; against /20 audit):

| Baseline | Setup | /20 |
|---|---|---:|
| ε | frozen Qwen3-235B + reference plan in context | 12.13 |
| **μ baseline** | plan-level SDPO + oracle v3 abstraction (10 iters, lr=1e-5, anchor_ce=0) | **8.88** |
| β | direct SFT on reference plan (10 SFT steps × 1 ex) | 7.75 |
| δ | frozen Qwen3-30B + reference plan in context | 6.75 |
| α | naive Opus distillation (5 epochs × 16 Opus plans) | 6.12 |

**Pairwise inversion discovery** (5 matchups × 8 random pairs):
- μ vs ε: 0–8 (ε wins all) — same direction as absolute
- **μ vs δ: 0–8 (δ wins all)** — INVERTED vs absolute (μ 8.88 > δ 6.75)
- μ vs β: 8–0 (μ wins all) — same direction
- α vs δ: 0–8 (δ wins all) — same direction
- β vs α: 4–4 tie

**Why δ wins pairwise**: δ has the *reference plan* in its prompt (semi-oracle), and Opus pairwise judges concrete substance (LoRA rank, β values, named benchmarks) over structural patterns. The +2.13 absolute lift for μ over δ was surface-form bias.

**Fair pairwise re-evaluation** (no reference plan in either side's prompt, 5 matchups):

| Matchup | Winner | Implication |
|---|---|---|
| smoke_v3_A vs smoke_v3_B | A 8-0 | Oracle inference HELPS |
| μ vs smoke_v3_A | A 6-2 | **SDPO training adds nothing over oracle inference** |
| μ vs smoke_v3_B | μ 8-0 | μ's lift is from oracle, not from training |
| α vs smoke_v3_B | smoke_B 5-2 | Opus distillation HURTS |
| β vs smoke_v3_B | smoke_B 6-2 | Ref SFT HURTS |

**Pre-realignment definitive truths** (formed the basis for the 2026-04-25 realignment decision):
1. Inference-time scaffolding works.
2. Plan-level SDPO at lr=1e-5 internalizes nothing.
3. Naive distillation methods (α, β) actively HURT.
4. The "+1.88 / +4.00 / +3.13 absolute audit lifts" had grader-bias issues but the underlying pathway (oracle helps inference) survived under fair pairwise.

**Paper-readiness verdict**: ARCHIVED. Do NOT cite. The realigned Phase 2 numbers supersede everything here. Mention only as the empirical motivation for the realignment.


### 4.4 The Realignment (2026-04-25 PM)

User identified four quality issues with all prior runs:
1. **Oracle abstraction was single-round** — derived from one reference plan only; not multi-round bibliography-grounded with citations.
2. **δ and ε were treated as baselines but are semi-oracle conditions** (they see the full reference plan in prompt).
3. **Audit prompts had surface-form bias** — favored Pattern X structure over concrete substance.
4. **No reviewer-standard grounding** for any audit dimension.

**Action**: archive everything, redesign the methodology from scratch with explicit reviewer-grounded standards.

**Locked artifacts** (in `knowledge/current/`):

| File | Purpose |
|---|---|
| `REVIEWER_STANDARDS_v1.md` | Synthesis of NeurIPS / ICLR / ICML / NSF reviewer guidelines + 7 OpenReview adjacent papers (MTTT, ReST-MCTS, ReST-EM, Voyager, SCoRe, Guided-ReST, DeepEvolve). Anchors used as basis for AUDIT_RUBRIC_v3 dimensions. |
| `ORACLE_DESIGN_v1.md` | Locked design: 6 categories (Insights / Methodology / Theory / Math / Empirical / Failure_modes), Opus-generated as ceiling estimate, multi-round (3 rounds) with cumulative memory, full bibliography fetch via S2 API, per-item citations. |
| `AUDIT_RUBRIC_v3.md` | Locked design: 9 dimensions hybrid (5 universal: Soundness / Significance / Originality / Clarity / Reproducibility + 4 TTT-Discover-specific: Necessity / Disentanglement / Compute / Reward-hacking), anchors calibrated to real OpenReview reviewer quotes. |
| `RUN_CONFIRMATION_TEMPLATE.md` | Pre-run user-confirmation gate. Every future run must fill this in and be signed off before launch. |

This is when the project went from "smoke iteration" to "paper-aimed".


### 4.5 Phase 2A-bis — bibliography rebuild from LaTeX source (2026-04-26)

**Why**: Phase 0b's PDF-based bibliography fetch silently dropped 17/87 references (pypdf extraction errors).

**Pipeline** (sequential — ~$0 because all metadata APIs):

| Stage | Script | I/O |
|---|---|---|
| 2A-bis-1 | `extract_cite_keys_v1.py` | main.tex → 90 dedupe'd cite keys (regex `\cite{...}` capturing arxiv 2601.16175v2's main.tex) |
| 2A-bis-2 | `parse_bibtex_v1.py` | main.bib (416 entries) + cite_keys → cited_bibtex_v1.jsonl (90 entries with arxiv_id + doi extracted) |
| 2A-bis-3..7 | `build_bibliography_v2.py` | S2 batch on arxiv → S2 batch on DOI → OpenAlex title search → arxiv PDF download → pypdf extract → **resolved_v2.jsonl (86 / 90 resolved)**, full_text/*.md (38 papers; 30K char limit) |
| 2A-bis-8 | `expand_full_text_v1.py` | 48 abstract-only entries → arxiv API title search → S2 externalIds → OpenAlex pdf_url → 4 reclassified as non-paper (yaoyour, openevolve, atcoder, white2023erdos partly) |
| 2A-bis-9 | `extract_arxiv_latex_v1.py` | arxiv e-print download → tar.gz extract → find main.tex → resolve_macros → inline \input → latex_to_markdown (extract_source_paper_v2.latex_to_markdown) → full_text_latex/*.md (28 LaTeX-→-MD converted) |
| 2A-bis-6 | `extract_source_paper_v2.py` | TTT-Discover main.tex → source_paper/v2.md (clean, ~76,300 chars; replaces corrupted PDF v1) |

**Final state**:
- `data/bibliography/resolved_v2.jsonl`: 90 entries, 86 with metadata, 38 with full text path, 4 marked non-paper, 4 unresolved (white2023erdos, etc).
- `data/bibliography/full_text/*.md`: 38 PDF-extracted markdowns (with pypdf artifacts).
- `data/bibliography/full_text_latex/*.md`: 28 LaTeX-extracted markdowns (clean, used preferentially when present).
- `data/source_paper/v2.md`: clean LaTeX-extracted TTT-Discover paper.

**Paper-readiness verdict**: PIPELINE ONLY (Methods section appendix). Don't dwell on the rebuild story — just note "bibliography resolved via S2 + OpenAlex with LaTeX-source full-text extraction".


### 4.6 Phase 2B — Oracle v2 build (2026-04-26)

**Hypothesis**: a multi-round Opus extraction with cumulative cross-paper memory produces a richer, more deduplicated, citation-traceable methodological oracle than the single-round v1.

**Pipeline** (`build_oracle_v2.py` orchestrator + `oracle_prompts_v2.py` prompts; ~$X Opus subagent cost — 4 rounds × 62 papers):

| Round | Prompt | Effect |
|---|---|---|
| 0 | `ROUND0_RELEVANCE_PROMPT` | Each paper scored 0/1/2/3 for relevance + expected categories. Filter: only score ≥1 retained → 62 papers from 90. |
| 1 | `ROUND1_EXTRACT_PROMPT` | Each retained paper extracted (0–3 items per category, full RHS for Math, concrete numbers for Empirical, named mechanisms for Methodology). |
| 2 | `ROUND_REFINE_PROMPT` | Each paper re-extracted with `accumulated_state` (round 1 over all papers) visible. Cross-paper deduplication; `action: keep / refine / new`. |
| 3 | `ROUND_REFINE_PROMPT` | Same as round 2 but with round-2 accumulated state. Final extraction. |

**Outputs**:
- `oracles/oracle_v2_2026_04_26_build/round_0..3_summary.json`: aggregated.
- `oracles/oracle_v2_2026_04_26_build/final.md`: 580 typed items in 6 categories (Insights 1–48, Methodology 1–97, Theory 1–36, Math 1–24, Empirical 1–22, Failure_modes 1–20). 2,922 lines.

**Slim and medium variants** (`build_slim_oracle_v2.py`):
- **slim** (`oracle_v2_slim.md`): score ≥3 only, ≤1 item per (paper, category). 17 papers, ~87 items, ~20K tokens. Fits Qwen3-30B-A3B 40K context. **Used by σ baseline + μ trainers v2/v3/v4.**
- **medium** (`oracle_v2_medium.md`): score ≥2, ≤1 item per (paper, category). ~181 items, ~30K tokens. **Used by τ Phase 3 distillation pipeline (3-batch split: (60, 60, 61)).**

**Verbatim prompts for oracle build** (Appendix A.6 below).

**Paper-readiness verdict**: METHODS SECTION DELIVERABLE. The 6-category structure + multi-round refinement + per-item citation traceability is one of D5's methodological contributions even if it's "Opus does the extraction".


### 4.7 Phase 2D — realigned baselines (2026-04-26)

Six baselines + the μ-v2 trainer were rebuilt against the locked v2 oracle. Each one tests an isolated hypothesis:

| Greek | Trainer | Setup | What it tests |
|---|---|---|---|
| ξ | `baseline_frozen_v2.py` (variant=xi) | Frozen Qwen3-30B-A3B + goal only | Floor — what the model can do with NO scaffolding |
| σ | `baseline_frozen_v2.py` (variant=sigma) | Frozen Qwen3-30B-A3B + goal + slim oracle | The key reference baseline (no training) |
| δ | `baseline_frozen_v2.py` (variant=delta) | Frozen Qwen3-30B-A3B + goal + reference plan in context | 30B's hard ceiling under frozen + privileged info |
| ε | `baseline_frozen_v2.py` (variant=epsilon) | Frozen Qwen3-235B-A22B + goal + reference plan in context | Absolute ceiling (235B + privileged info) |
| α | `train_alpha_v2.py` | Opus distillation: SFT 5 epochs × 16 Opus plans, with oracle in prompt at SFT and EVAL | Tests "is Opus distillation enough?" |
| β | `train_beta_v2.py` | Direct SFT on reference plan, 10 iters × 1 ex, oracle in prompt | Tests "is reference imitation enough?" |
| μ-v2 | `train_mu_v2.py` | Plan-level (off-policy IS-loss) "SDPO" + Opus critic + oracle in prompt | First realigned trainer; ablates lr=1e-5 |

**Common config** (from source code, file:line in Appendix B):
- Model: Qwen3-30B-A3B (Tinker).
- LoRA rank=64, alpha unspecified in code — uses Tinker default (64 or 128).
- Sampling: temperature=1.0, top_p=0.95, max_tokens=4096.
- Oracle path: `data/oracle_v2_2026_04_26_build/slim.md`.
- Plan target: 600 words, max 750 (per `_PLAN_FOOTER` in `mu_prompts_v2.py`).

**Paper-readiness verdict**: BASELINES TABLE DELIVERABLE. All 7 baselines (incl. μ-v2) plus production μ-v4 form the 8-baseline matrix in `paper_materials/findings/F1_sigma_oracle_lift.md` and the F2 production result.


### 4.8 Phase 2F — audit_v3 ISOLATED methodology (2026-04-26)

**Hypothesis**: per-plan independent scoring with anonymization eliminates within-batch templating bias that daemon audit and audit_v3_runner exhibited.

**Method** (`audit_v3_isolated.py`):
- 8 baselines × 8 plans each = 64 total.
- Build 8 balanced batches: each batch contains 1 plan from each of the 7 baselines (8 plans per batch including the test condition). Wait — actually 8 batches × 7 plans (1 per baseline) — see source code line 145–148.
- Each batch shuffled and anonymized as `plan_A` … `plan_G`. The mapping is saved post-hoc to `decode_map.json`.
- Each of 8 batches is dispatched to **a dedicated Opus 4.7 subagent** via the file-bus subagent client.
- Each subagent instructed: "score each plan INDEPENDENTLY on the 9-dim rubric. Do NOT batch-template — read each plan's specific content and score on what's actually there. Two plans from the same source can score very differently if their content differs."
- Each subagent outputs `{anon_id → {per_dim_scores, total_raw, key_differentiator}}`.
- After all 8 subagents return, the `decode_map.json` is used to attribute scores back to baselines.

**Rubric**: 9 dimensions × 0–5 each → /45 total. Universal (5 dims) + TTT-Discover-specific (4 dims). Full anchor table in Appendix A.7.

**Why this is the gold standard**:
- Daemon audit (Phase 0) used 4 dimensions and saturated at the anchor (F6: scores all μ-v4 iters identically at 7.00/20).
- audit_v3_runner (`audit_v3_runner.py`) bundled all 8 plans of each baseline into one Opus session → templating bias (Opus calibrates batch-internally and gives every plan in batch similar scores).
- audit_v3_shuffled (`audit_v3_shuffled.py`) put all 56 plans in one shuffled batch → still anchored across the 56 plans.
- audit_v3 ISOLATED (this) gives each batch its own Opus subagent + each plan an anonymized ID → cross-baseline templating prevented.

**Paper-readiness verdict**: METHODOLOGY SECTION DELIVERABLE. Documented in `paper_materials/methodology/M4_audit_isolated_methodology.md`.


### 4.9 Phase 2 — 7-baseline ranking under audit_v3 ISOLATED (2026-04-26)

**Verified scores** (cross-checked against `runs/2026_04_26_phase2F_audit_isolated/audit_v3_isolated_summary.md` raw data):

| Baseline | Setup | Mean /45 | vs σ |
|---|---|---:|---:|
| ε | frozen Qwen3-235B + ref in context | **33.62** ✅ | +8.37 |
| **σ** | **frozen Qwen3-30B + slim oracle** | **25.25** ✅ | — |
| δ | frozen Qwen3-30B + ref in context | 25.12 ✅ | −0.13 |
| α | Opus distillation 5×16 | 25.00 ✅ | −0.25 |
| **μ-v2** | **plan-level SDPO + critic + oracle, 10 iters lr=1e-5** | **23.88** ✅ | **−1.37** |
| β | direct SFT on reference plan | 22.75 ✅ | −2.50 |
| ξ | frozen Qwen3-30B + goal only | 15.62 ✅ | −9.63 |

**F1 finding** (paper-grade): the σ baseline (frozen + oracle) is +9.63 over the floor (ξ). The single largest effect in the entire baseline matrix. Source paper material: `paper_materials/findings/F1_sigma_oracle_lift.md`.

**Implication for μ training**: μ-v2 LOSES to σ by 1.37 points at audit_v3 ISOLATED. Plan-level SDPO at lr=1e-5 internalizes nothing (mid-training PUCT diagnostic: PICK 0/8 → 8/8 over iters; EVAL stays 0/8). The training-method comparison is what produced the lr ablation (Phase 2 → μ-v3 → μ-v4).

**Paper-readiness verdict**: TABLE-1 DELIVERABLE. F1 is the strongest finding by effect size; F2 (μ-v4 +2.75 over σ) is the headline contribution.


### 4.10 Phase 2 — μ-v2/v3/v4 SDPO learning-rate ablation (2026-04-26 → 2026-04-27)

Common knobs (from `train_mu_v2.py` / `train_mu_v3.py` / `train_mu_v4.py`):

| Knob | μ-v2 | μ-v3 | μ-v4 (production) |
|---|---:|---:|---:|
| `learning_rate` | 1e-5 | 2e-4 | **5e-5** ≈ √(1e-5 × 2e-4) |
| `n_iter` | 10 | 20 | 20 (peak iter 4) |
| `n_grad_steps_per_iter` | 1 | 4 | 4 |
| `n_plans` (per iter, EVAL pool) | 8 | 8 | 8 |
| `max_tokens` (plan generation) | 4096 | 4096 | 4096 |
| `temperature` | 1.0 | 1.0 | 1.0 |
| `top_p` | 0.95 | 0.95 | 0.95 |
| `lora_rank` | 64 | 64 | 64 |
| `sdpo_scale` | 1.0 | 1.0 | 1.0 |
| `sdpo_clip_advantage` | 5.0 | 5.0 | 5.0 |
| `anchor_ce_weight` | 0.0 | 0.0 | 0.0 (Phase 2; >0 only in Phase 5α) |
| `eval_every` | 2 | 3 | **1 (every iter)** |
| `save_every` | 5 | 3 | **1 (every iter, for rollback)** |
| `audit_drop_threshold` | n/a | n/a | **3.0 (early-stop guard)** |

**Loss formulation (identical across μ-v2..v4 — verified at file:line in Appendix B)**:
```
For each plan k in 8:
  TEACHER samples plan_k under prompt = goal + oracle + critique_prev.
  STUDENT logprobs are recomputed under prompt = goal + oracle (no critique).
  per_token_advantage_t = clamp( sdpo_scale * (teacher_lp_t - student_lp_t),
                                  -sdpo_clip_advantage, +sdpo_clip_advantage )
  Build a Tinker `Datum` with:
    model_input = student_prompt_tokens + sampled_plan_tokens
    target_tokens = sampled_plan_tokens (shifted)
    logprobs = student_lp (per-token, on student prefix)
    advantages = per_token_advantage
  Append to datums.
training_client.forward_backward(datums, loss_fn="importance_sampling")
training_client.optim_step(adam_params)  # lr above
```

This is the off-policy IS-loss D5 in-house variant — TEACHER samples on its prompt, STUDENT logprobs recomputed under its prompt, IS update applied to STUDENT parameters on TEACHER's tokens. **NOT canonical Hübotter SDPO (which is on-policy + KL); NOT canonical Zhao OPSD (which is on-policy + JS or sampled-token PG).**

**Critic mechanism** (synchronous per iter, all three trainers identical):
- Per iter, after sampling 8 plans, pick a random one.
- Build a `build_critic_request_payload()` with goal, source paper, plan text.
- Submit to OpusCriticClient (file-bus subagent, 900s timeout).
- Wait for response → parse `<critique>...</critique>` XML.
- Use this critique as `prev_critique` for next iter's TEACHER prompt.
- If critic times out: fall back to `COLD_START_CRITIQUE_NONE`.

**Result** (verified raw audits):

| Trainer | lr | Trajectory (best /45 in early iters) | Outcome |
|---|---:|---|---|
| μ-v2 | 1e-5 | flat ~23.88 across all iters | NO LEARNING — gradient budget too small |
| μ-v3 | 2e-4 | iter-0 24.88 → iter-6 14.38 | CATASTROPHIC COLLAPSE (Chinese-character bleed, biocultural headers, thinking-mode JSON leaks). User: "kill 吧" |
| **μ-v4** | **5e-5** | iter 0=24.00, iter 1=24.75, iter 2=24.88, iter 3=27.25, **iter 4=28.00**, iter 5=15.62, iter 6=~9.6, iter 7=~9.8 | **PRODUCTION** — peak iter 4. Iter-5 cliff -12.4 in 1 iter |

**Production checkpoint**: `runs/2026_04_27_mu_v4/checkpoints.jsonl` row `batch=4`. SDPO advantage signal stayed healthy throughout (`mean_adv ≈ 0.5, pos_frac ≈ 80%`).

**SDPO_RECIPE_v1.md** (locked in `knowledge/current/`) freezes: lr=5e-5, n_iter=20 (expect peak iter 3-5), n_grad_steps=4, lora_rank=64, sdpo_scale=1.0, clip=5.0, eval_every=1, save_every=1, audit_drop_threshold=5.0, loss_fn="importance_sampling".

**Paper-readiness verdict**: F2 IS THE HEADLINE QUANTITATIVE RESULT. Caveats:
1. Single goal (TTT-Discover only).
2. n=8 plans per iter; std ~3-4. Pairwise corroboration is the cross-validation that justifies the claim.
3. Multi-round instability is real (iter 5 cliff). Production is iter-4 checkpoint, not iter-7.
4. Terminology: do not call this "SDPO" without disambiguation (see §2.1).


### 4.11 Phase 2 — pairwise corroboration of μ-v4 (2026-04-27)

**Why**: the audit gap σ → μ-v4 is +2.75/45 < 5/45. Per the M8 standing protocol, close-cluster gaps require pairwise tournament cross-validation.

**Method** (`mu_v4_pairwise_v1.py`):
- 3 matchups: μ-v4 vs σ (PRIMARY, ≥6/8 PASS threshold), μ-v4 vs δ, μ-v4 vs α.
- Each matchup: 8 random pairs, position-randomized A/B (~50% swap rate).
- Judge: Opus 4.7 via OpusPairwiseClient file-bus subagent.
- Strict 1 subagent per pair (per the 2026-04-27 task-isolation rule).

**Verified outcome** (raw data in `runs/2026_04_27_mu_v4_pairwise/`):

| Matchup | μ-v4 wins | Baseline wins | Tally | A-pos rate | Verdict |
|---|---:|---:|---|---:|---|
| μ-v4 vs σ (PRIMARY) | 6 | 2 | 6-2 | 50% | ✅ PASS |
| μ-v4 vs δ | 7 | 1 | 7-1 | 62% | (wide margin) |
| μ-v4 vs α | 7 | 1 | 7-1 | 75% | (within bound) |
| **Total** | **20** | **4** | **20/24 (83.3%)** | — | ✅ |

**Why this corroborates the headline**: the same audit_v3 ISOLATED methodology that detected μ-v4 +2.75 also gives σ +0.13 over δ — the pairwise reveals μ-v4 wins close-cluster matches at high rate, confirming the audit signal is real not artifact.

**Paper-readiness verdict**: F2 PAPER-GRADE. Both audit /45 and pairwise /8 evidence go in main text.


### 4.12 Phase 3 — τ inference-time distillation pipeline (2026-04-27)

**Reframe** (locked 2026-04-27 via Q3): Phase 3 originally "retrieve-then-generate" (replace static slim oracle with per-section dynamic retrieval over 86-paper bibliography) was reframed to "**dynamic distillation pipeline**" — Path Y. Reasoning: medium oracle (181 items, 30K tokens) is small enough to feed in 3 batches without learned retrieval; the question becomes "does iterating distillation+plan generation help over single-shot σ?".

**τ_v1 → τ_v2 → τ_v3 → τ_v4 evolution**:

| Variant | Change | Why |
|---|---|---|
| τ_v1 | Original 3-round distillation, max_tokens=1024 | Phase 3 baseline |
| τ_v2 | max_tokens 1024 → **3500** | Qwen3 thinking mode (~500–1000 tok) was eating the entire budget; A_1 was 100% thinking with 0 distillation |
| τ_v3 | (unstated tweaks; superseded) | — |
| τ_v4 | + plan footer v4 (mandatory T1/T2/T3 directives + citation hygiene) | Address audit lift via prompt directives, not algorithm |

**τ_v4 pipeline** (`train_tau_v1.py`, frozen — NO training, NO LoRA, NO critic):
1. Per instance i ∈ {1..8}, sample 3 distillations independently:
   - Round 1: build_distillation_student_prompt(goal, oracle_batch_1, round_idx=1) → distillation A_1.
   - Round 2: build_distillation_student_prompt(goal, oracle_batch_2, round_idx=2) → distillation A_2 (no shared state with round 1 — see `_DISTILL_FOOTER` "rounds do not share state").
   - Round 3: same with oracle_batch_3 → A_3.
   - Oracle batches: medium oracle's 181 items split into (60, 60, 61) batches with seed=42.
2. Build plan prompt: build_kappa_plan_prompt(goal, [A_1, A_2, A_3], version="v4") = goal + 3 distillation sections + `_KAPPA_PLAN_FOOTER_V4`.
3. Sample 1 plan per instance via frozen Qwen3-30B-A3B.

**F8 v1 (RETRACTED 2026-04-27 PM)**:
- Original headline: τ_v4 cross-instance mean = 26.50 / 45, **wins 7-1 STRONG vs σ_v4** (Step 2.5 pairwise).
- Step 2a BoN sanity (K=8 reseed at instance 0): mean 24.88 vs production single-shot 27 → **gap 2.12, σ_within = 1.17** = LUCKY DRAW.
- Step C (instance 2 K=4 σ spot-check): σ_within = 0.50 (different from inst 0's 1.17, ratio 2.3× heterogeneous).
- Step E pairwise rerun (8 fresh pairs, position-balanced, seed=52): **4-4 TIE** (50% A-pos).
- → **F8 v1 RETRACTED**.

**F8 v2** (revised; verified):
- τ_v4_clean (citation-hygiene patched, fresh sample) cross-instance mean = **24.50 / 45**, σ across 8 = 2.18, 95% CI [22.99, 26.01]. Verified raw scores: [26, 21, 28, 24, 23, 27, 24, 23].
- σ_v4 (control) = 25.75 / 45. CI overlap.
- Pairwise τ_v4_clean vs σ_v4 = 4-4 TIE.

**3 characterized failure modes** (now the substantive Phase 3 contribution):
1. **M7 self-review degeneration**: Qwen3-30B-A3B as self-critic 0-8 lost to Opus critic on same 8 plans (Smoke A `smoke_a_critique_correlation_v1.py`). Privileged-info comprehension gap.
2. **Single-shot evaluation inflation**: σ_within ≈ 1.17 means single-shot evaluation cannot distinguish 24.50 from 26.50.
3. **Citation-tag leakage**: Distillation organizes oracle by index labels ("Methodology 17", "Math 11"); plan generator copies these as if real citations. Patched ~85% in `_KAPPA_PLAN_FOOTER_V4` "Citation hygiene" instruction; full fix deferred.

**Phase 3 paper-grade claim** (REVISED per F8 v2): "open-source 30B inference-time pipeline produces plans **statistically comparable** to hand-curated baseline (no fine-tuning)". 3-round distillation is NOT a positive substitute for SDPO training. **Negative-result-with-mechanism — ICLR-publishable.**

**κ deferred**: κ-v1 smoke (cold-start critic, single iter) audit = 24.88 — explicitly INVALIDATED 2026-04-27 ("cost-saving compromise; SDPO advantage on cold-start text is not the same test as real critic"). κ-opus full smoke deferred (~$60-100 cost, low priority now that τ_v4 ≈ σ).

**Paper-readiness verdict**: F8 v2 IS A LEGITIMATE NEGATIVE RESULT. Not a headline win, but a properly-characterized failure with mechanism — exactly the kind of result ICLR should accept.


### 4.13 Phase 4a — cross-goal validation (2026-04-26 → 2026-04-28)

**Hypothesis**: μ-v4 LoRA weights (trained on TTT-Discover) transfer to forward-citation follow-up papers that share the same TTT-Discover topic — i.e., the LoRA learned "general TTT-style plan quality" rather than overfitting to one paper.

**Test design** (pre-registered 2026-04-26):
- 3 forward-citation papers (avoiding backward refs to prevent oracle leakage):
  - **Meta-TTL** (Lou 2604.00830).
  - **TT-Control** (Wang 2603.09221).
  - **Tool-Verification-TTRL** (Liao 2603.02203).
- Inference-only (no retraining): generate σ' (frozen + slim oracle) and μ' (μ-v4 LoRA + slim oracle) plans for each goal.
- Slim oracle is reused VERBATIM (TTT-Discover bibliography); user explicitly chose forward-citation papers because their bibliography overlaps with TTT-Discover's so the existing oracle has coverage.
- Audit: audit_v3 ISOLATED 2-way (`audit_v3_isolated_2way.py`), 8 balanced batches × 2 plans (1 σ' + 1 μ').

**Phase 4a result table** (verified):

| Goal | σ' /45 | μ' /45 | Δ | Pass (≥+1.0)? |
|---|---:|---:|---:|---|
| meta_ttl | 17.75 ✅ | 19.88 ✅ | +2.12 | ✅ |
| tt_control | 20.00 ✅ | 19.12 ✅ | -0.88 | ❌ |
| tool_v_ttrl | 20.38 ✅ | 23.38 ✅ | +3.00 | ✅ |

**Original verdict (2026-04-26)**: "Strong-mixed (2/3 pass)" per pre-registered matrix.

**Phase 4a remediation** (2026-04-27): the user invoked the "M8 close-cluster pairwise corroboration" rule. Re-audited via:
- **Strict 1-plan/subagent** (`audit_v3_isolated_2way_strict.py`): 16 batches × 1 plan, anonymized, no within-batch anchoring.
- **Pairwise** (`phase5_remediation_pairwise.py`): 8 random pairs per goal.

**Cross-protocol comparison** (2026-04-27):

| Goal | 2-plan Δ (orig) | Strict 1-plan Δ | Pairwise tally | Verdict |
|---|---:|---:|:---:|---|
| meta_ttl | +2.13 | -0.12 | σ' 5/3 | **2-plan inflated; strict + pairwise say σ' edge** |
| tt_control | -0.88 | -0.63 | σ' 5/3 | 3-protocol convergence (σ' edge) |
| tool_v_ttrl | +3.00 | -0.63 | μ' 4/3/1 (TIE) | **2-plan inflated; pairwise+strict ≈ TIE** |

→ **F7 DOWNGRADED**: "Strong-mixed 2/3 pass" → "all 3 inconclusive within audit noise; meta_ttl direction-reverses under pairwise; methodology lesson (M8) is the surviving contribution".

**Phase 4a stress test (2026-04-28)** — D-series n=24:
- meta_ttl n=24: strict Δ -0.96, pairwise σ' 14/9/1 (58%) → σ' DIRECTIONAL CONFIRMED at n=24.
- tool_v_ttrl n=24: strict Δ -0.58, pairwise σ' 12/10/2 (50%) → TRUE NULL at n=24.

**F7 verdict (final, 2026-04-28)**: **DEFINITIVELY NULL** at n=24 across both goals. μ-v4 LoRA does NOT transfer to forward-citation follow-up papers. Original "Strong-mixed" was a 2-plan/subagent batch anchoring artifact (M8).

**Surviving contributions**:
1. F7 as a properly-characterized negative result (n=24 stress-tested).
2. **M8 methodology contribution** (Phase 6 below).
3. Per-dim signature M8_per_dim: μ-v4 LoRA wins on U5 / T3 / T4 — exactly the dims it was empirically trained to lift (compute, reproducibility, reward-hacking awareness). Loss is on T1/T2 (necessity, disentanglement) which are goal-specific factors not trainable with single-paper oracle.

**Paper-readiness verdict**: F7 IS A SUBSTANTIAL NEGATIVE FINDING. Don't paper over it. The strongest framing is "single-paper SDPO produces a goal-shaped LoRA, not a transfer-ready 'general planning' LoRA — and the audit-pairwise divergence (M8) is what made the negative result detectable."


### 4.14 Phase 5 — continual SDPO 4-cell grid (2026-04-26)

**Hypothesis (H1)**: Continual SDPO from a μ-v4 anchor checkpoint to a new paper (Tool-V) maintains forward learning. I.e., there exists a configuration where v5 isolated /45 > μ-v4 isolated /45 by ≥1.0 on Tool-V.

**Design** — 3-axis diagnostic:
1. **Anchor saturation**: μ-v4 iter 4 (saturated) vs μ-v4 iter 2 (mid-training).
2. **Optimizer state**: full Adam (inherit moments) vs fresh Adam (weights only).
3. **anchor_ce_weight**: 0 (pure SDPO) vs 0.1 (regularization toward TTT-D anchor).

**4 cells run** (`train_mu_v4.py` with `init_state_path` set to μ-v4 checkpoint):

| Run | Anchor | Adam | anchor_ce | Best /20 in-loop | Failure mode |
|---|---|---|---:|---:|---|
| cliff_v1 | μ-v4 iter 4 | full | 0 | 8.38 | iter-1 cliff -3.50 |
| cliff_v2 | μ-v4 iter 4 | fresh | 0 | 9.00 | iter-1 cliff -2.13 |
| v3 (iter2anchor) | μ-v4 iter 2 | fresh | 0 | 9.375 | iter 0→4 slow decline -2.71 |
| α (anchor_ce) | μ-v4 iter 2 | fresh | **0.1** | 10.375 | iter-2 cliff -3.0 (regularization backfires) |

**Audit verification on Tool-V** (audit_v3 ISOLATED, 2026-04-27):
- v3 (iter2anchor) Δ vs μ-v4 = +0.62 (within audit noise ±1.76 at n=8); strict +1.87; pairwise 4-4 TIE.
- α Δ = -2.62; strict +0.26 (anchor_ce verdict softened to TIE under strict, but trajectory cliff at iter-2 remains the load-bearing failure signal).

**F9 finding (NULL across 4 cells, triple-corroborated)**:
- All 4 cells fail H1 forward learning (best v5_v3 +0.62 within noise; α -2.62 with anchor_ce backfire).
- Failure SHAPES vary (immediate cliff vs delayed vs slow decline) but never SUCCESS.
- 3-protocol agreement (in-loop trajectory + audit /45 + pairwise) on NULL.

**Mechanistic hypothesis** (CONTINUAL_SDPO_FAILURE_MODES_v1.md): stylistic-token gradient mass dominates (~70% of tokens). Two competing gradients (TTT-D anchor + Tool-V SDPO) fight → incoherent middle. anchor_ce regularization (when present) compounds by pulling toward TTT-D distribution, accelerating collapse on Tool-V signal.

**Mitigations identified** (Phase 6α candidates, NOT YET RUN):
1. Plan-level replay buffer (`train_mu_v6_replay.py` — implemented, not deployed in Phase 5).
2. EWC-Fisher penalty.
3. LoRA-per-paper merge (don't continual-train; train one LoRA per goal, merge at inference).

**Phase 4b cross-domain stretch test FORMALLY DEPRECATED** (2026-04-26): given Phase 5 NULL, no point in running cross-domain (climate / bio / econ / chem) continual SDPO until naive continual is fixed.

**Paper-readiness verdict**: F9 IS THE MOST RIGOROUSLY VALIDATED NEGATIVE FINDING in the project (4-cell grid + 3-protocol agreement). Should anchor the paper's narrative on SDPO limitations: "single-paper SDPO works (F2); continual SDPO across papers fails by 4-cell grid exhaustion (F9)".


### 4.15 Phase 6 — M8 methodology contribution (2026-04-27)

**Discovery**: 2-plan/subagent batching (the original Phase 4a `audit_v3_isolated_2way.py` design) introduces a systematic anchoring bias that inflates close-cluster Δ by ~2–3 points relative to strict 1-plan or pairwise.

**Mechanism**: when 2 plans (1 σ', 1 μ') sit in the same Opus subagent's view, the subagent calibrates them against each other rather than against its full reference distribution. Whichever plan has a slight edge gets that edge amplified by within-batch anchoring.

**Validation** — 5 goals × 3 protocols (80 strict subagents + 32 pairwise subagents + 1 batch_13 JSON-fix):

| Goal | 2-plan Δ | Strict 1-plan Δ | Pairwise | Verdict |
|---|---:|---:|:---:|---|
| meta_ttl | +2.13 | -0.12 | σ' 5/3 | 2-plan inflated 2.25 pts |
| tt_control | -0.88 | -0.63 | σ' 5/3 | 3-protocol convergence |
| tool_v_ttrl | +3.00 | -0.63 | TIE 4/3/1 | 2-plan inflated 3.6 pts |
| Phase 5 v3 | +0.62 | +1.87 | TIE 4/4 | NULL within noise |
| Phase 5 α | -2.63 | +0.26 | n/a | strict softens but trajectory cliff is real |

**Standing protocol (paper-grade rule)**:
- Cross-tier comparisons (e.g., ε vs σ at +8.37): 2-plan or batched audit acceptable; pairwise corroborates.
- Within-tier close-cluster comparisons (Δ < ~5/45): **strict 1-plan/subagent OR pairwise tournament REQUIRED**.

**Phase 6 7-baseline pairwise validation** (`phase6_seven_baseline_pairwise.py` — 3 close-cluster matchups: σ vs δ, σ vs α, δ vs α):
- σ vs δ: σ wins 6-2 (audit underestimated; audit Δ +0.13).
- σ vs α: 3-5 with A-pos rate 88% — position bias outside [25%, 75%]. INCONCLUSIVE.
- δ vs α: 3-5 with A-pos rate 88% — INCONCLUSIVE.

**M8 cross-goal effect partially generalizes**: the σ vs δ matchup confirms 2-plan audit underestimated σ's edge. Position-bias issues on σ vs α / δ vs α are M8 caveats but the general principle (close-cluster requires pairwise) stands.

**Paper-readiness verdict**: M8 IS A REAL METHODOLOGY CONTRIBUTION (a "what evaluation works for long-form generation" finding). Documented in `paper_materials/methodology/M8_audit_pairwise_divergence.md`. Eligible for the methods-novel section of the paper.


### 4.16 Phase 7 — μ-v7 OPD canonical-friendly architectural fix (2026-04-27 → 2026-04-29)

**Motivation** (CANONICAL_NAMING_REFERENCE.md, 2026-04-27): the D5 in-house off-policy IS-loss variant (μ-v2..v6) is *not* canonical SDPO. To check whether canonical on-policy gradients would change the F3/F4 instability story or unlock cross-goal transfer (F7), build μ-v7-opd.

**Design** (`train_mu_v7_opd.py`):
- `opd_mode=True`: STUDENT samples on-policy under (goal+oracle); TEACHER logprobs are recomputed under (goal+oracle+critique).
- Same advantage formula `A_t = clamp(scale*(teacher_lp − student_lp), [-clip, clip])`, but on-manifold (student samples its own distribution).
- Per-token IS-loss (NOT canonical reverse-KL — Tinker exposes only per-token logprobs, not full-vocab logits).
- Closest existing analog: Zhao OPSD's sampled-token policy gradient variant (Table 3, ~2% gap to canonical OPSD).

**Verified results**:

| Metric | μ-v4 (off-policy) | μ-v7-opd (on-policy) |
|---|---:|---:|
| Iter 4 audit /45 | 28.00 | 26.625 |
| Iter 5 audit /45 | 15.62 | 23.125 (cliff delayed +2 iters) |
| Combined iters 4+5 mean | 21.81 | **24.875** |
| Pairwise (μ-v7-opd vs μ-v4 iter 4) | — | **8/8 wins** |
| Cross-goal (3 forward-citation papers, audit /45 mean) | NULL | meta_ttl 22.875, tool_v_ttrl 23.75, tt_control 23.75 → aggregate **23.458**, still NULL |

**F13 finding** (μ-v7-opd canonical architecture):
- μ-v7-opd wins pairwise vs μ-v4 8/8 on TTT-D; cliff delayed 2 iters; J_β oracle content recovered (μ-v4 0/8, μ-v7-opd 3/8 at iter 5).
- HOWEVER, peak audit /45 LOWER than μ-v4 (26.625 vs 28.00). The TTT-D evaluation may favor the off-policy variant on a goal-specific stylistic surface but the on-policy variant generalizes better across iters.

**F14 finding** (cross-goal still NULL): the canonical-friendly architecture does NOT rescue cross-goal transfer (xgoal aggregate 23.458 < σ on the same goals). The cross-goal failure is not a (c)-HER artifact; it is a deeper limitation of single-paper SDPO of any flavor.

**Decision deferred for paper**: highlight μ-v4 (established headline +2.75) OR μ-v7-opd (canonical-friendly architecture with delayed-cliff and pairwise dominance). Both are defensible; the wrong move is conflating them.

**Paper-readiness verdict**: F13/F14 ARE PAPER-RELEVANT. The cleanest paper might (a) report μ-v4 as the headline +2.75, (b) include μ-v7-opd as architecture-ablation showing on-policy doesn't fix cross-goal, (c) note in CANONICAL_NAMING that "SDPO" in D5 is an in-house off-policy IS-loss variant — full transparency.


## 5. Component deep-dives

This section is a structural reference for the paper's Methods section.

### 5.1 Training pipeline (μ-v4 production)

**Per iter (i = 0 .. 19)**:

```
# Step 1 — sample 8 plans under TEACHER context
For k in 1..8:
  teacher_prompt = build_teacher_prompt(goal, oracle, prev_critique)
  plan_k = teacher_sampling_client.sample(
      prompt = teacher_prompt,
      num_samples = 1,
      sampling_params = SamplingParams(temperature=1.0, top_p=0.95, max_tokens=4096),
  )

# Step 2 — recompute STUDENT logprobs on TEACHER's tokens
For k in 1..8:
  student_prompt = build_student_prompt(goal, oracle)  # NO critique
  student_lp_k = sampling_client.compute_logprobs(
      ModelInput.from_ints(tokens = student_prompt + plan_k.tokens)
  )
  # slice student_lp_k[len(student_prompt):len(student_prompt) + len(plan_k.tokens)]

# Step 3 — build per-token advantage
For k, t:
  A_t = clamp(sdpo_scale * (plan_k.logprobs[t] - student_lp_k[t]), [-5, +5])

# Step 4 — build SDPO datums
For k:
  datum_k = Datum(
    model_input = ModelInput.from_ints(tokens = student_prompt + plan_k.tokens[:-1]),
    loss_fn_inputs = {
      target_tokens: plan_k.tokens (shifted),
      logprobs: student_lp_k[:-1],
      advantages: A_k[:-1],
    },
  )

# Step 5 — microbatch grad steps
chunks = split_into_4_chunks(datums)
For chunk in chunks:
  fb = training_client.forward_backward(chunk, loss_fn="importance_sampling")
  os = training_client.optim_step(adam_params)  # lr=5e-5, weight_decay=0.0

# Step 6 — Opus critic dispatch (synchronous, blocking)
chosen_idx = rng.randrange(8)
chosen_plan = per_plan_records[chosen_idx]
payload = build_critic_request_payload(iter, goal, chosen_plan, source_paper, plan_id)
critic_client.submit(iter, payload)
resp = critic_client.wait(iter, timeout=900s)
prev_critique = parse_critique_xml(resp) or COLD_START_CRITIQUE_NONE  # fallback if timeout

# Step 7 — eval (8 plans under STUDENT prompt, no critique) + audit dispatch
For e in 1..8:
  eval_prompt = build_student_prompt(goal, oracle)
  eval_plan_e = sampling_client.sample(eval_prompt, num_samples=1, ...)
audit_client.submit_async(iter, eval_plans)  # do NOT block

# Step 8 — early-stop guard
if rolling_mean(last_2_audits) - current_audit > audit_drop_threshold:
  log "EARLY STOP", break
```

**Critical mechanism** — TEACHER samples ⇒ STUDENT supervised on TEACHER's tokens ⇒ off-policy IS-loss update applied to STUDENT parameters. Full-vocab logits NOT exposed by Tinker, so this is per-token scalar logprob, NOT canonical reverse-KL distillation.

**Cost estimates**: 20 iters × (8 plans × ~30s sampling + 1 critic call ~60-120s + 8 audit ~120s + 4 grad steps) ≈ 4-5 hours wall on Tinker. Per-iter Opus cost: ~$0.80 (1 critic) + ~$3 (8 audits) ≈ $4. Full run ≈ $80 Opus.


### 5.2 Inference-time τ pipeline (no training)

**Per instance (i = 1 .. 8)**:

```
# Round j = 1, 2, 3 (independent — no cross-round state!)
For j in 1..3:
  oracle_batch_j = oracle_v2_medium[batch_j]  # 60-61 items
  prompt_j = build_distillation_student_prompt(goal, oracle_batch_j, round_idx=j)
  raw_distillation_j = frozen_qwen3_30b.sample(prompt_j, max_tokens=3500, ...)
  A_j = extract_distillation(raw_distillation_j)  # strip Qwen3 thinking + <solution>...</solution>

# Plan generation
prompt = build_kappa_plan_prompt(goal, [A_1, A_2, A_3], version="v4")
plan_i = frozen_qwen3_30b.sample(prompt, max_tokens=4096, ...)
```

**Key prompt** — `build_kappa_plan_prompt`:
- Inputs: goal + 3 distillation lists.
- Output footer (v4): mandatory T2 (disentanglement ablation), T3 (compute accounting in operational units), T1 (quantified frozen-LLM baseline gap), and citation-hygiene warning.

**Cost**: per instance ~3 Qwen3 calls (free on Tinker) + 1 plan call. 8 instances → 32 Tinker calls (~5 min wall). Audit cost: 1 Opus subagent batch ($1-2). **τ_v4 inference is free; the cost is the audit.**


### 5.3 Audit pipeline (audit_v3 ISOLATED — gold standard)

**Per phase**:

```
# Inputs: dict[baseline_name, list[plan_text]] with 8 plans each
N_BASELINES = 7 or 8  (depending on phase)
PLANS_PER_BASELINE = 8

# Build 8 balanced batches (random round-robin)
batches = []
For b in 1..8:
  batch = []
  For baseline in baselines:
    batch.append(baselines[baseline][b - 1])  # b-th plan of each baseline
  shuffle(batch)
  anonymize: plan_A, plan_B, ..., plan_G (or whatever count)
  batches.append((batch, decode_map[b]))

# Save decode_map.json (anon_id → (baseline, plan_idx))

# Dispatch each batch to a dedicated Opus subagent
For batch_idx in 1..8:
  request_payload = build_audit_v3_isolated_request(
    goal, batch_plans, prompt_version="v3_isolated",
  )
  # The prompt instructs:
  #   "score each plan INDEPENDENTLY on the 9-dim rubric.
  #    Do NOT batch-template — read each plan's specific content
  #    and score on what's actually there. Two plans from the
  #    same source can score very differently if their content differs."
  audit_subagent_dispatch(request_payload)  # Opus 4.7 file-bus

# Wait for all 8 batches to return
For batch_idx in 1..8:
  response = audit_subagent_wait(batch_idx)
  per_plan_scores[batch_idx] = parse(response)  # {anon_id: {U1:5, ..., T4:3, total:30}}

# Decode back to baselines
For baseline:
  totals_per_plan = [decode_map_lookup(b, anon_id) for b in 1..8 ...]
  mean_per_baseline[baseline] = mean(totals_per_plan)
```

**Strict 1-plan variant** (`audit_v3_isolated_2way_strict.py`): 16 single-plan batches × 1 plan/batch. Each plan gets its own Opus subagent. The prompt: *"This is ONE anonymized research plan. You don't know which method produced it. Score it on the 9-dim rubric using ABSOLUTE calibration..."*

**Pairwise variant** (`mu_v4_pairwise_v1.py`, `phase5_remediation_pairwise.py`, `step_e_pairwise_v1.py`):
- Per matchup, 8 pairs, position-randomized A/B (~50% swap).
- Each pair → 1 Opus subagent (post-2026-04-27 strict task isolation).
- Subagent prompt: *"Read goal then plan A and plan B carefully. Judge which is the better research plan on dimensions: methodological rigor, originality, reproducibility, concrete-spec depth, grounding in cited prior work, disentanglement design, compute realism, reward-hacking awareness. Be sharp — do not default to ties unless plans are truly indistinguishable."* Output: `{"winner": "A" | "B" | "TIE", "rationale": "<2-3 sentences>"}`.
- Tally rule: ≥6/8 STRONG, 5/8 WEAK, 4/8 TIE.


## 6. Verbatim prompts (Appendix A)

Every prompt appears below verbatim from source code. File:line citations included so you can verify each one.

### A.1 — Plan generation footers

**`mu_prompts_v1.py:29-33` — `_PLAN_FOOTER`** (used by μ baseline_v1):
```
\n\nWrite your research plan inside <solution>...</solution> tags. Target 600 words, max 750 words. Use structured sections (Problem Statement, Background, Hypothesis, Methodology, Evaluation, Limitations).
```

**`mu_prompts_v2.py:33-37` — `_PLAN_FOOTER`** (used by μ-v2/v3/v4, σ-v2, δ-v2, ε-v2, ξ-v2, α-v2, β-v2):
```
\n\nWrite your research plan inside <solution>...</solution> tags. Target 600 words, max 750 words. Use structured sections (Problem Statement, Background, Hypothesis, Methodology, Evaluation, Limitations).
```

**`kappa_prompts_v1.py:152-161` — `_KAPPA_PLAN_FOOTER`** (used by τ_v1 / κ_v1 with version="v3"):
```
\n\nWrite your research plan inside <solution>...</solution> tags. Target 600 words, max 750 words. Use structured sections (Problem Statement, Background, Hypothesis, Methodology, Evaluation, Limitations). The three distilled abstraction lists above were produced independently from disjoint oracle batches; expect overlap between them. INTEGRATE the lists, DEDUPLICATE shared items, and prioritize items most relevant to the research goal. Do not list the abstractions in your plan — use them as grounding.
```

**`kappa_prompts_v1.py:166-197` — `_KAPPA_PLAN_FOOTER_V4`** (added 2026-04-27; used by τ_v4 + κ when version="v4"):
```
\n\nWrite your research plan inside <solution>...</solution> tags. Target 600 words, max 750 words. Use structured sections (Problem Statement, Background, Hypothesis, Methodology, Evaluation, Limitations). The three distilled abstraction lists above were produced independently from disjoint oracle batches; expect overlap between them. INTEGRATE the lists, DEDUPLICATE shared items, and prioritize items most relevant to the research goal. Do not list the abstractions in your plan — use them as grounding.

**Citation hygiene**: The distilled abstractions above use internal index labels like "Methodology 17", "Theory 5", "Math 11", "Algorithm 3", "Background 2" — these are bookkeeping tags from the source organization, NOT real citations. Use the conceptual content but DO NOT reproduce these tags in your plan. Cite real prior work by first-author + year (e.g., "Hübotter 2026") or by descriptive name only.

**MANDATORY plan-quality elements** (these are not in the distilled abstractions; you must compose them yourself):
- **Disentanglement ablation** (Methodology or Evaluation section): name 1-2 specific ablations that separate the LLM-prior contribution from the method's contribution (e.g., 'frozen-LLM same-prompt baseline', 'method without per-problem updates', 'random-init LoRA'). Cite which plan component each ablation isolates.
- **Compute accounting in operational units** (Evaluation or Limitations section): state compute as GPU-hours per problem (or wall-clock minutes per problem), NOT as FLOP-only or memory-only. Give a concrete number range (e.g., '2-4 H100-hours per problem').
- **Quantified frozen-LLM baseline gap** (Background or Evaluation): state the prior-art frozen-LLM number on at least one named benchmark (e.g., 'frozen GPT-4 scores 50.4% on MATH-500') AND your method's expected delta with rationale.
```

**`train_sigma_v4_v1.py:78-99` — `_SIGMA_V4_FOOTER`** (used by σ_v4 in Phase 3):
```
\n\nWrite your research plan inside <solution>...</solution> tags. Target 600 words, max 750 words. Use structured sections (Problem Statement, Background, Hypothesis, Methodology, Evaluation, Limitations).\n\n**MANDATORY plan-quality elements**:\n- **Disentanglement ablation** (Methodology or Evaluation section): name 1-2 specific ablations that separate the LLM-prior contribution from the method's contribution (e.g., 'frozen-LLM same-prompt baseline', 'method without per-problem updates', 'random-init LoRA'). Cite which plan component each ablation isolates.\n- **Compute accounting in operational units** (Evaluation or Limitations section): state compute as GPU-hours per problem (or wall-clock minutes per problem), NOT as FLOP-only or memory-only. Give a concrete number range (e.g., '2-4 H100-hours per problem').\n- **Quantified frozen-LLM baseline gap** (Background or Evaluation): state the prior-art frozen-LLM number on at least one named benchmark (e.g., 'frozen GPT-4 scores 50.4% on MATH-500') AND your method's expected delta with rationale.
```


### A.2 — STUDENT / TEACHER prompts (μ training and σ baseline)

**`mu_prompts_v2.py` re-exports from v1; STUDENT prompt (no critique)** — used by σ baseline + μ EVAL channel:

```python
def build_student_prompt(goal: str, oracle_abstraction: str) -> str:
    return (
        "I will provide you a research scenario and a set of methodological"
        " patterns extracted from a high-quality research plan on this same"
        " scenario. Use the patterns to guide the structure and reasoning of"
        " your plan, but do not copy the phrasing literally."
        f"\n\nScenario: {goal}"
        f"\n\n# Methodological patterns to apply\n\n{oracle_abstraction}"
        + _PLAN_FOOTER
    )
```

**`mu_prompts_v1.py:53-70` — TEACHER prompt (with prior critique)** — used by μ training PICK channel:

```python
def build_teacher_prompt(goal: str, oracle_abstraction: str, critique_xml: str) -> str:
    return (
        "I will provide you a research scenario, a set of methodological"
        " patterns extracted from a high-quality research plan on this same"
        " scenario, and a structured critique of a prior plan attempt. Use the"
        " patterns to guide structure and the critique's improvement_directive"
        " to address concrete weaknesses. Do not copy phrasing literally."
        f"\n\nScenario: {goal}"
        f"\n\n# Methodological patterns to apply\n\n{oracle_abstraction}"
        f"\n\n# Critique of prior attempt (to address)\n\n{critique_xml}"
        + _PLAN_FOOTER
    )
```

**`mu_prompts_v1.py:81-88` — `COLD_START_CRITIQUE_NONE`** (fallback when iter-0 has no prior plan, or critic times out):

```xml
<critique>
<idea_alignment>This is the first iteration; no prior plan exists to align against. Focus on producing a well-derived plan that addresses the methodological patterns provided.</idea_alignment>
<missing_components>To be evaluated against the source paper after generation. Ensure each pattern's derivation steps are followed, not skipped.</missing_components>
<incorrect_assumptions>To be evaluated against the source paper after generation. State assumptions explicitly so they can be checked.</incorrect_assumptions>
<feasibility>State a concrete compute budget, a concrete target hardware, and a concrete evaluation-baseline number even if estimated. Avoid hand-waving on numerical claims.</feasibility>
<improvement_directive>Derive each pattern's training/search objective from first principles (do not skip the derivation Steps). Name a concrete compute budget. State 1-3 specific evaluation baselines with prior numerical results. Name 3-5 mechanism-level limitations (not implementation limitations). Avoid generic phrases like "we will use standard reinforcement learning" — specify the loss form, the search procedure, and the stability mechanism.</improvement_directive>
</critique>
```

**`mu_prompts_v1.py:95-126` — `build_critic_request_payload()` instructions field** (sent to Opus critic subagent per iter):

```
You are an expert research-methodology reviewer. You have privileged access to the source paper. Use it as ground truth when judging idea alignment, missing components, incorrect assumptions, and feasibility. DO NOT mention the source paper by name, by author, or by arxiv ID in your output. Write a structured <critique> block with the following children: <idea_alignment>, <missing_components>, <incorrect_assumptions>, <feasibility>, <improvement_directive>. Each is 1-3 sentences except improvement_directive (≤120 words). Wrap the entire response inside <critique>...</critique>. Output nothing else.
```

**`mu_prompts_v2.py:40-57` — `build_alpha_v2_distill_prompt()` (Opus distillation prompt)**:

```python
def build_alpha_v2_distill_prompt(goal: str, oracle: str) -> str:
    return (
        "You are an expert ML/AI research methodologist. Write a complete"
        " research plan for the scenario below, drawing on the methodological"
        " patterns provided. Your plan will be used as a distillation target"
        " for a smaller open-weight model, so be specific: name mechanisms,"
        " include equations with full RHS where relevant, and cite concrete"
        " numerical baselines for any claims about prior work."
        f"\n\nScenario: {goal}"
        f"\n\n# Methodological patterns to apply\n\n{oracle}"
        + _PLAN_FOOTER
    )
```

**`train_alpha_v2.py:109-123` — α-v2 student prompt** (training and eval — same prompt):

```python
def build_student_prompt(goal: str, oracle: str) -> str:
    return (
        "I will provide you a research scenario and a set of methodological"
        " patterns extracted from a high-quality research plan on this same"
        " scenario. Use the patterns to guide the structure and reasoning of"
        " your plan, but do not copy the phrasing literally."
        f"\n\nScenario: {goal}"
        f"\n\n# Methodological patterns to apply\n\n{oracle}"
        + _PLAN_FOOTER
    )
```

**`train_alpha_baseline_v1.py:106-114` — α baseline (no oracle) student prompt**:

```python
def build_student_prompt(goal: str) -> str:
    return (
        "I will provide you a research scenario. You have to provide me a"
        " concise yet thoughtful research plan with all details needed to"
        " execute it."
        f"\n\nScenario: {goal}"
        + _PLAN_FOOTER
    )
```


### A.3 — Distillation prompts (κ / τ Phase 3)

**`kappa_prompts_v1.py:83-103` — `_DISTILL_FOOTER`** (used by κ STUDENT and TEACHER distillation prompts):

```
\n\nWrite your distilled item list inside <solution>...</solution> tags. Target 800-1200 tokens. Use the SAME 6 categories as the input oracle (Insights / Methodology / Theory / Math / Empirical / Failure_modes). For each item kept: write 1-3 sentences (longer if the item contains concrete equations or numerical content). **PRESERVE equations, hyperparameter values, and named benchmark numbers VERBATIM in backticks** (e.g. `β=8`, `LoRA rank 64`, `AIME pass@1 = 53.4%`, `J = log E[exp(β·R)]/β`); only rewrite surrounding prose. Aim for 40-50 distilled items across all categories combined (this batch); **PRIORITIZE items containing equations or numerical specifics over items containing only abstract principles**. Drop oracle items that are clearly orthogonal to the research goal.

# Example of GOOD vs BAD distillation
## GOOD (preserves substance):
- **RS-GRPO** (Math): `J_RS = log E[exp(β·R)] / β` with `β=8` stabilizes test-time policy updates by emphasizing high-reward rollouts (Hubotter et al. 2025). Applies to TTT-Discover's per-problem RL loop.
## BAD (substance lost):
- Use risk-sensitive RL methods to focus on high-reward solutions.
```

**`kappa_prompts_v1.py:106-122` — `build_distillation_student_prompt()`**:

```python
def build_distillation_student_prompt(
    goal: str, oracle_batch_text: str, round_idx: int
) -> str:
    return (
        "You are extracting goal-relevant methodological abstractions from a"
        " curated oracle of insights drawn from prior work. This is round"
        f" {round_idx} of 3 independent distillation rounds (rounds do not"
        " share state — focus only on this batch)."
        f"\n\n# Research goal\n{goal}"
        f"\n\n# Oracle batch (round {round_idx})\n\n{oracle_batch_text}"
        + _DISTILL_FOOTER
    )
```

**`kappa_prompts_v1.py:125-145` — `build_distillation_teacher_prompt()`**:

```python
def build_distillation_teacher_prompt(
    goal: str, oracle_batch_text: str, critique_xml: str, round_idx: int
) -> str:
    return (
        "You are extracting goal-relevant methodological abstractions from a"
        " curated oracle of insights drawn from prior work. This is round"
        f" {round_idx} of 3 independent distillation rounds. A reviewer with"
        " privileged access to the source paper has critiqued a prior"
        " distillation attempt; address the improvement_directive while"
        " selecting and condensing items from this batch."
        f"\n\n# Research goal\n{goal}"
        f"\n\n# Oracle batch (round {round_idx})\n\n{oracle_batch_text}"
        f"\n\n# Critique of prior attempt (to address)\n\n{critique_xml}"
        + _DISTILL_FOOTER
    )
```

**`kappa_prompts_v1.py:200-229` — `build_kappa_plan_prompt()`** (final plan from 3 distillations):

```python
def build_kappa_plan_prompt(goal: str, A_list: list[str], version: str = "v3") -> str:
    if len(A_list) != 3:
        raise ValueError(f"expected 3 distillations, got {len(A_list)}")
    sections = "\n\n".join(
        f"## Distillation round {i} (independent)\n\n{a}"
        for i, a in enumerate(A_list, 1)
    )
    base = (
        "You will write a research plan for the scenario below, grounded in"
        " three distilled abstraction lists. Each list was produced"
        " independently from a disjoint subset of the methodological oracle,"
        " so they may overlap or partially contradict. Integrate carefully."
        f"\n\n# Research goal\n{goal}"
        f"\n\n# Distilled abstractions (3 independent rounds)\n\n{sections}"
    )
    if version == "v4":
        return base + _KAPPA_PLAN_FOOTER_V4
    return base + _KAPPA_PLAN_FOOTER
```

**`kappa_prompts_v1.py:236-242` — `DISTILLATION_COLD_START_CRITIQUE`** (round-1 fallback in κ-v1):

```xml
<distill_critique>
<missing_critical>This is round 1; no prior distillation exists. Focus on extracting items that name concrete mechanisms (loss forms, search procedures, stability bounds) and items whose Applicability section directly engages the research goal. Drop items whose Source field cites work obviously orthogonal to the goal.</missing_critical>
<noise>To be evaluated against the source paper after this round. Avoid items whose Content section is purely empirical (numbers without mechanism) unless they set a critical baseline number.</noise>
<faithfulness>To be evaluated against the source paper after this round. When condensing, preserve key technical terms (loss names, algorithm names, hyperparameter symbols) verbatim; rewrite surrounding prose only.</faithfulness>
<improvement_directive>For each kept item, write 1-2 sentences condensing both the mechanism AND the applicability to the research goal. Prefer items that name a specific algorithm or equation over items that describe a general principle. Avoid hand-waving phrasings like "this approach has been shown to work well" — name what it produces.</improvement_directive>
</distill_critique>
```

**`kappa_prompts_v1.py:249-295` — `build_distillation_critic_request_payload()` instructions** (sent to Opus distillation-critic):

```
You are an expert research-methodology reviewer with privileged access to the source paper. The text below is a DISTILLED abstraction list — a subset of methodological items selected from a larger oracle. Judge whether this distillation captures what the source paper says is core for the research goal. DO NOT mention the source paper by name, by author, or by arxiv ID in your output. Write a structured <distill_critique> block with the following children: <missing_critical> (≤3 sentences naming items the source paper emphasizes that the distillation missed), <noise> (≤3 sentences naming items the distillation included that the source paper does not consider central), <faithfulness> (≤3 sentences naming any item that the distillation paraphrased incorrectly relative to the original oracle item's content), <improvement_directive> (≤120 words, free-text guidance for the next distillation attempt). Wrap the entire response inside <distill_critique>...</distill_critique>. Output nothing else.
```

**`kappa_prompts_v1.py:323-364` — `build_self_review_critic_prompt()`** (Qwen3-30B base self-critic, used in `train_kappa_v1_self.py`):

```python
def build_self_review_critic_prompt(
    *, goal: str, distillation_text: str, source_paper: str, round_idx: int,
) -> str:
    return (
        "You are an expert research-methodology reviewer with privileged"
        " access to the source paper for the research goal below. The text"
        " after `# Distilled abstraction list` is a candidate distillation"
        " produced by another model — your job is to critique it against the"
        " source paper as ground truth. Round " + str(round_idx) + " of 3"
        " (the next round will use your critique to revise)."
        "\n\n# Research goal\n" + goal +
        "\n\n# Source paper (privileged ground truth — for your reasoning"
        " only; DO NOT mention the paper by name, author, or arxiv ID in"
        " your critique)\n" + source_paper +
        "\n\n# Distilled abstraction list to critique\n" + distillation_text +
        "\n\n# Output format"
        "\nWrite a structured <distill_critique> block with these children:"
        "\n- <missing_critical>: ≤3 sentences naming items the source paper"
        " emphasizes that the distillation missed"
        "\n- <noise>: ≤3 sentences naming items the distillation included"
        " that the source paper does not consider central"
        "\n- <faithfulness>: ≤3 sentences naming any item the distillation"
        " paraphrased incorrectly relative to the original oracle item"
        "\n- <improvement_directive>: ≤120 words concrete guidance for the"
        " next distillation attempt (specific items to add, items to drop,"
        " preservation rules)"
        "\n\nWrap the entire response inside <distill_critique>...</distill_critique>."
        " Output nothing else."
    )
```


### A.4 — Pairwise judge prompts

**`pairwise_v2.py:59-74` — Prompt Variant 1 (PV1, minimal)**:

```
You are evaluating two research plans on the same scientific problem. Pick the better
one or call it a TIE.

# Research Goal
{goal}

# Plan A
{plan_a}

# Plan B
{plan_b}

Output JSON only:
{"winner": "A|B|TIE", "rationale": "<1 sentence>"}
```

**`pairwise_v2.py:76-95` — Prompt Variant 2 (PV2, reviewer-lite)**:

```
You are evaluating two research plans on the same scientific problem. Pick the better
one or call it a TIE. Consider:
- Soundness: are claims supported by math or empirical evidence?
- Novelty: is the proposed approach distinct from cited prior work?
- Feasibility: are compute, hyperparameters, and evaluation realistic?
- Empirical rigor: are baselines named with prior numbers; are stats/seeds specified?

# Research Goal
{goal}

# Plan A
{plan_a}

# Plan B
{plan_b}

Output JSON only:
{"winner": "A|B|TIE", "rationale": "<1-2 sentences referencing the criteria>"}
```

**`pairwise_v2.py:97-135` — Prompt Variant 3 (PV3, full 9-dim)**:

```
You are an expert ML/AI conference reviewer comparing two research plans on the same
scientific problem. First score each plan on 9 dimensions (1-5 each), then output a
verdict based on the per-dim differences.

# Dimensions

UNIVERSAL (1-5 each):
- U1 Soundness: claims supported by evidence
- U2 Significance: real problem with real impact
- U3 Originality: novel beyond pretraining-knowledge
- U4 Clarity: well-reasoned and structurally clear
- U5 Reproducibility: compute / hparams / stats specified

SUBFIELD (1-5 each, for test-time-search / search-with-LLMs papers):
- T1 Necessity: shows test-time RL is necessary
- T2 Disentanglement: separates parameter updates from search alone
- T3 Compute accounting: latency / cost honesty
- T4 Reward-hacking awareness: probes for shortcuts/Goodhart

# Research Goal
{goal}

# Plan A
{plan_a}

# Plan B
{plan_b}

# Output

Output JSON only with this schema:
{
  "scores_a": {"U1":<1-5>, "U2":<1-5>, ..., "T4":<1-5>},
  "scores_b": {"U1":<1-5>, ..., "T4":<1-5>},
  "winner": "A|B|TIE",
  "rationale": "<2-3 sentences referencing the most decisive per-dim differences>"
}
```

**`step_e_pairwise_v1.py:118-127` — Step E pairwise judge prompt** (used for F8 v1 retraction):

```
You are an Opus 4.7 pairwise research-plan judge. Read the goal, then plan A and plan B carefully. Judge which is the better research plan on the dimensions: methodological rigor, originality, reproducibility, concrete-spec depth (formulas/hparams/baselines), grounding in cited prior work, disentanglement design, compute realism, reward-hacking awareness. Be sharp — do not default to ties unless plans are truly indistinguishable. Output JSON: {"winner": "A" | "B" | "TIE", "rationale": "<2-3 sentences citing specific differences>"}.
```


### A.5 — audit_v3 ISOLATED instructions (per-batch subagent)

**`audit_v3_isolated.py:171-179`** (in-prompt instructions for the dedicated Opus subagent per batch):

```
This is a batch of 7 anonymized research plans (plan_A..plan_G). They came from 7 different methods on the same scientific problem, but you are NOT told which plan came from which method.

Score each plan INDEPENDENTLY on the 9-dim rubric (5 universal + 4 subfield). Do NOT batch-score — read each plan's specific content carefully. Two plans from the same source can score very differently if their content differs.

For each plan, output:
- per-dim scores (U1..U5 + T1..T4, 1-5 each)
- total /45
- 1-2 sentences "key_differentiator" — the most distinctive aspect that drove the score

Output JSON only with this schema:
{
  "plan_A": {"U1": <1-5>, ..., "T4": <1-5>, "total_raw": <sum>, "key_differentiator": "..."},
  ...,
  "plan_G": {...}
}
```

**`audit_v3_isolated_2way_strict.py:135-144` — Strict 1-plan/subagent variant** (used for Phase 5 remediation):

```
This is ONE anonymized research plan. You don't know which method produced it. Score it on the 9-dim rubric (5 universal + 4 subfield) using ABSOLUTE calibration — read the plan's specific equations, named benchmarks, mechanisms, failure modes and score on what's actually there. There is no other plan to compare against in this batch — calibrate against your full internal reference distribution of ML research plans for this research-goal subfield.
```


### A.6 — Oracle extraction prompts (Phase 2B)

**`oracle_prompts_v2.py:46-83` — `ROUND0_RELEVANCE_PROMPT`** (relevance filter):

```
You are evaluating whether a research paper is relevant for grounding a research plan
on the following research goal.

# Research Goal
{goal}

# Candidate Paper
- Title: {title}
- Authors: {authors_short}
- Year: {year}
- Venue: {venue}
- Abstract: {abstract}
- TLDR: {tldr}

# Task

Score this paper's relevance to the research goal on a 0-3 scale, then identify which of
the 6 oracle categories (Insights, Methodology, Theory, Math, Empirical, Failure_modes)
this paper would contribute to.

Score meaning:
- 0 = baseline citation only (e.g. classic textbook reference, citation for a foundational
  concept). Provides no methodological/conceptual contribution to a plan on THIS goal.
- 1 = relevant context but secondary (provides background but doesn't suggest specific
  reusable mechanisms for this plan).
- 2 = primary methodology / theory / empirical source for at least one category.
- 3 = core source — Opus would recommend reading the full paper to extract specific
  formulas, operators, or numerical results.

# Output Format

Respond with ONLY this JSON (no other text, no markdown fence):

{"relevance_score": <0|1|2|3>,
 "rationale": "<1-2 sentences explaining the score>",
 "expected_categories": [<subset of "Insights"|"Methodology"|"Theory"|"Math"|"Empirical"|"Failure_modes">]}
```

**`oracle_prompts_v2.py:90-151` — `ROUND1_EXTRACT_PROMPT`** (cold-start extraction):

```
You are extracting reusable insights from a research paper to ground a research plan
on the following goal.

# Research Goal
{goal}

# Source Paper
- Title: {title}
- Authors: {authors_short}
- Year: {year}
- Venue: {venue}
- Content: {content}

# Expected Categories (from earlier relevance filter)
{expected_categories_csv}

# Categories Reference

You will extract 0-3 items per category. Categories:

{category_descriptions}

# Task

For EACH category in `expected_categories` (and any other category if you find genuinely
reusable content), extract 0-3 items. Each item is a specific, concrete reusable element.

Constraints:
- "Math" items MUST contain a full RHS equation (e.g. `L = -E[r · log π]`) when present in
  the paper. If no equation, leave Math empty.
- "Empirical" items MUST cite specific numerical results from the paper (e.g. "60.4%
  accuracy on MATH-500"). If no concrete numbers, leave Empirical empty.
- "Methodology" items name a SPECIFIC mechanism (e.g. "PUCT selection rule",
  "rejection sampling fine-tuning") with the deterministic operator if available.
- Avoid vague entries like "use reinforcement learning" — be specific.

# Output Format

Respond with ONLY this JSON (no other text, no markdown fence):

{
  "Insights": [
    {"summary": "<≤10 word title>", "content": "<2-4 sentences>", "applicability": "<1-2 sentences on how it's reusable for the goal>"}
  ],
  "Methodology": [
    {"summary": "...", "content": "...", "applicability": "..."}
  ],
  "Theory": [...],
  "Math": [
    {"summary": "<formula name>", "content": "<formula with RHS + variables>", "applicability": "..."}
  ],
  "Empirical": [
    {"summary": "<benchmark or eval setting>", "content": "<concrete numbers>", "applicability": "..."}
  ],
  "Failure_modes": [
    {"summary": "<what failed>", "content": "<why + attribution>", "applicability": "<how this informs Limitations or Hypothesis>"}
  ]
}

If a category has NO content from this paper, set it to an empty list `[]`.
```

**`oracle_prompts_v2.py:158-213` — `ROUND_REFINE_PROMPT`** (rounds 2-3 with cumulative state):

```
You are refining the oracle abstraction. You see (a) the accumulated state from prior
rounds (other papers' extractions), (b) one paper's content, and (c) your prior-round
extraction for this paper.

The goal of refinement is to:
- Add new items if this paper contributes things not yet captured
- Sharpen existing items (more concrete content, better attribution)
- Mark obsolete items if a different paper covered the same point better

# Research Goal
{goal}

# Accumulated State (from prior round, all papers, organized by category)
{accumulated_state}

# This Paper
- Title: {title}
- Authors: {authors_short}
- Content: {content}

# Your Prior-Round Extraction for This Paper
{prior_extraction}

# Categories Reference

{category_descriptions}

# Task

Output the FINAL extraction for this paper after considering the accumulated state. May:
- Keep an item from prior extraction unchanged
- Refine an item (better content, deduplicate)
- Drop an item if already covered by another paper better
- Add a new item discovered on re-read

Same constraints as Round 1: Math items have full RHS; Empirical items have concrete
numbers; Methodology items name specific mechanisms.

# Output Format

Respond with ONLY this JSON (no other text, no markdown fence):

{
  "Insights": [
    {"summary": "...", "content": "...", "applicability": "...", "action": "keep"|"refine"|"new"}
  ],
  "Methodology": [...],
  "Theory": [...],
  "Math": [...],
  "Empirical": [...],
  "Failure_modes": [...]
}

Empty list `[]` if no items in a category.
```


### A.7 — Audit rubric v3 (anchors)

**Universal layer (5 dims × 1-5 each → /25)**:

```
U1. Soundness — Are claims supported by evidence (math or empirical)?
- 1: Claims unsupported (no equations, no benchmark numbers, no derivation)
- 2: Some claims supported but key claims hand-waved (e.g. method described but no formula)
- 3: Major claims supported by either inline math (RHS-complete) or named benchmark numbers — but not both
- 4: Claims supported by both inline math AND specific empirical numbers; minor hand-waving acceptable
- 5: All non-trivial claims supported with rigor; assumptions stated; robustness discussed

U2. Significance — Is this a real problem with real impact?
- 1: Toy problem, no clear practical value
- 2: Real problem but plan addresses only narrow slice
- 3: Real problem, plan addresses central question, but unclear it advances state of the art
- 4: Real problem, plan claims SOTA-comparable result with credible argument
- 5: Real problem, plan would advance SOTA demonstrably and be picked up by community

U3. Originality — Novel beyond pretraining-knowledge or trivial recombination?
- 1: Standard toolkit recombination ("REINFORCE + LoRA + buffer") with no insight
- 2: Minor adaptation of known method; some novelty in application
- 3: Non-obvious combination with at least one mechanism-level insight
- 4: Substantive modification with clear differentiation from prior work; new operator or new objective
- 5: Genuinely novel algorithm or framing; clear theoretical/empirical hook differentiating from cited prior work

U4. Clarity — Is the plan well-reasoned and structurally clear?
- 1: Disorganized; key sections missing or vague
- 2: Sections present but mechanisms unspecified ("apply policy gradient", "use small learning rate")
- 3: Mechanisms named but reasoning chain has gaps
- 4: Mechanisms specified with rationale; reasoning chain coherent end-to-end
- 5: Clear non-specialist-readable plan; each step justified; no jargon without explanation

U5. Reproducibility — Are compute, hyperparameters, code/data, and statistics specified?
- 1: No mention of compute, hyperparameters, statistical methodology
- 2: 1 of {compute, hparams, eval methodology} mentioned vaguely
- 3: 2 of 3 mentioned with some specificity
- 4: All 3 stated with concrete values (e.g. compute budget in $/GPU-hr; LoRA rank=N; n=K seeds)
- 5: All stated + statistical methodology (CIs, p-values, multi-seed aggregation) + open-model commitment
```

**TTT-Discover-specific layer (4 dims × 1-5 each → /20)**:

```
T1. Necessity of framing — Does the plan demonstrate that test-time RL is *necessary*, not just sufficient?
- 1: Plan does not address whether test-time RL is needed at all
- 2: Plan asserts test-time RL helps but no comparison to in-context-only baseline
- 3: Plan mentions a frozen-LLM baseline but doesn't quantify gap
- 4: Plan specifies a frozen-LLM best-of-N baseline with prior numbers and explains why test-time RL exceeds it
- 5: Plan explicitly characterizes the regime where test-time RL is *necessary* (e.g. specific reward sparsity, specific compute budget) and where it would NOT help

T2. Disentanglement — Does the plan separate LLM-prior contribution from method contribution?
- 1: Plan treats LLM+method as one black box
- 2: Plan acknowledges base-model dependence but no ablation proposed
- 3: Plan proposes one of {frozen baseline / different base model / pretraining-only baseline}
- 4: Plan proposes ≥2 disentanglement ablations with expected results
- 5: Plan proposes full disentanglement matrix (frozen vs adapted × different bases) with explicit hypothesis on which factor dominates

T3. Compute / cost accounting in operational units
- 1: No compute statement
- 2: Compute mentioned in non-operational units (e.g. "compute-efficient" or just FLOP)
- 3: Compute stated in one operational unit (e.g. GPU-hours or $)
- 4: Compute stated in multiple operational units with method-specific accounting (e.g. $X for training, $Y per inference, ~K rollouts at $Z each)
- 5: Full operational accounting + sensitivity analysis (e.g. "cost scales linearly with X; halving Y doubles cost")

T4. Reward-hacking / saturation analysis
- 1: No analysis of reward-objective robustness or iteration-limit
- 2: Acknowledges potential reward hacking or saturation but no concrete pathway analysis
- 3: Names ≥1 specific Goodhart pathway OR ≥1 specific saturation mechanism; one of the two
- 4: Both analyzed: Goodhart pathways identified + saturation behavior characterized; mitigations proposed
- 5: 4 + the plan includes an explicit adversarial-probe experiment or a stop-criterion based on saturation detection
```

Total **/45**.

Cross-domain variants in `knowledge/current/`:
- `AUDIT_RUBRIC_v3_climate.md` (Hébert & Laepple, climate fingerprint)
- `AUDIT_RUBRIC_v3_phyloformer2.md` (Blassel et al., likelihood-free phylogenetic inference)

Each variant keeps U1-U5 verbatim and adapts T1-T4 to the source paper's structure.


## 7. Hyperparameter matrix (Appendix B)

All trainers, hyperparams as in source code. The "Loss" column documents whether the algorithm is canonical SDPO (no — none of D5 is), canonical OPSD-SPG (closest is μ-v7-opd), or D5 in-house off-policy IS-loss (μ-v2..v6, κ).

| Trainer | Source file | Algorithm | LR | n_iter | n_grad/iter | n_plans | LoRA rank | Loss | Critic | Anchor CE | Replay |
|---|---|---|---:|---:|---:|---:|---:|---|---|---:|---|
| α v1 | `train_alpha_baseline_v1.py` | Opus distillation SFT | 1e-5 | 5 epochs | — | 16 (Opus plans) | 64 | importance_sampling | none | — | — |
| α v2 | `train_alpha_v2.py` | Opus distillation SFT + oracle | 1e-5 | 5 epochs | — | 16 | 64 | importance_sampling | none | — | — |
| β v1 | `train_beta_baseline_v1.py` | Reference SFT | 1e-5 | 10 | — | 1 (ref plan) | 64 | importance_sampling | none | — | — |
| β v2 | `train_beta_v2.py` | Reference SFT + oracle | 1e-5 | 10 | — | 1 | 64 | importance_sampling | none | — | — |
| κ v1 (smoke) | `train_kappa_v1.py` | D5 in-house off-policy IS (distillation level, cold-start critic) | 5e-5 | 2 | 4 | 8 inst × 3 rounds | 64 | importance_sampling | cold-start | — | — |
| κ v1 self | `train_kappa_v1_self.py` | D5 in-house off-policy IS (distillation level, Qwen3 self-critic) | 5e-5 | 3 | 4 | 8 × 3 | 64 | importance_sampling | Qwen3-30B base | — | — |
| μ baseline_v1 | `train_mu_baseline_v1.py` | D5 in-house off-policy IS (plan level) | 1e-5 | 10 | 1 | 8 | 64 | importance_sampling | Opus subagent | 0.0 | — |
| μ v2 | `train_mu_v2.py` | Same, oracle_v2 slim | 1e-5 | 10 | 1 | 8 | 64 | importance_sampling | Opus subagent | 0.0 | — |
| μ v3 | `train_mu_v3.py` | Same + microbatch grad steps | **2e-4** | 20 | 4 | 8 | 64 | importance_sampling | Opus subagent | 0.0 | — |
| **μ v4** (production) | `train_mu_v4.py` | Same + early-stop guard + Phase 5α support | **5e-5** | 20 | 4 | 8 | 64 | importance_sampling | Opus subagent | 0.0 (Phase 2) / >0 (Phase 5α) | — |
| μ v6 replay | `train_mu_v6_replay.py` | Same + plan-level replay buffer | 5e-5 | 20 | 4 | 8 fresh + 4 replay | 64 | importance_sampling | Opus subagent | 0.0 | from prior μ-v* run |
| μ v7 OPD | `train_mu_v7_opd.py` (`opd_mode=True`) | **D5 in-house ON-POLICY IS** (closest to Zhao OPSD-SPG) | 5e-5 | 20 | 4 | 8 | 64 | importance_sampling | Opus subagent | 0.0 | — |
| σ v4 | `train_sigma_v4_v1.py` | Frozen + slim oracle + plan footer v4 | — | — | — | 8 | — | — | none | — | — |
| τ v1/v4 | `train_tau_v1.py` | Frozen + 3-round distillation pipeline | — | — | — | 8 inst × 3 rounds | — | — | none | — | — |

**SDPO advantage formula** (μ v2..v6, κ — and identical in form for μ v7 OPD with sources swapped):
```
A_t = clamp( sdpo_scale * (teacher_lp_t - student_lp_t),
             -sdpo_clip_advantage, +sdpo_clip_advantage )

# Default: sdpo_scale=1.0, sdpo_clip_advantage=5.0
```

**μ v6 replay step**:
```
For each iter, after fresh SDPO step:
  1. sample 4 plans from replay_buffer (prior runs' buffer.jsonl, filtered iter ∈ allowed_iters, len ≥ min_n_tokens)
  2. recompute student_lp under CURRENT weights
  3. build datums with constant_advantage = replay_anchor_weight (default 0.05)
  4. n_replay_steps microbatch grad steps
```

**μ v4 / v6 / v7 early-stop guard**: if rolling_mean(last 2 audits) − current_audit > audit_drop_threshold (default 3.0–5.0): halt, save final checkpoint.


## 8. Data verification report (Appendix C)

15 headline claims from `STATUS.md` were checked against raw `runs/*/audit_responses/*.json`. Results:

| # | Claim | Status | Notes |
|---|---|---|---|
| 1 | μ-v4 trajectory 24.00 / 24.75 / 24.88 / 27.25 / 28.00 / 15.62 / ~9.6 / ~9.8 | ⚠ scale-mixed | raw audits in iter directories use /36 internally; the 8-baseline summary table already uses /45. The `audit_v3_isolated_summary.md` reports the right /45 scale. The trajectory numbers cited in STATUS match the latter. Reading raw files outside the summary requires knowing which scale is in use. |
| 2 | σ = 25.25/45 | ✅ exact | n=8, [23, 22, 23, 28, 27, 26, 29, 24] / 8 = 25.25 |
| 3 | ε = 33.62/45 | ✅ exact | n=8, [33, 36, 32, 32, 32, 32, 37, 35] / 8 = 33.625 |
| 4 | δ=25.12, α=25.00, μ-v2=23.88, β=22.75, ξ=15.62 | ✅ all exact | every baseline matches summary table |
| 5 | Pairwise μ-v4 vs σ/δ/α: 6-2 / 7-1 / 7-1 = 20/24 | ✅ exact | summary table verified per-pair |
| 6 | Phase 4a meta_ttl Δ +2.12, tt_control -0.88, tool_v_ttrl +3.00 | ✅ exact | all three goals' summary files |
| 7 | D-series n=24 meta_ttl 14/9/1, tool_v_ttrl 12/10/2 | ✅ exact | 48 pairwise responses, both goals |
| 8 | Phase 5 best /20: cliff_v1 8.38, cliff_v2 9.00, v3 9.375, α 10.375 | ⚠ INCONCLUSIVE | the run dirs have training logs but I did not locate the per-iter /20 audit summary in the audit_responses files I checked. Likely computed in training metrics. |
| 9 | τ_v4 F8 v1 = 26.50 (RETRACTED) and F8 v2 = 24.50 (CI [22.96, 26.04]) | ✅ exact for v2 | v2 raw [26, 21, 28, 24, 23, 27, 24, 23] / 8 = 24.50; std=2.18; CI 24.50 ± 1.96·(2.18/√8) = [22.99, 26.01] (vs claim [22.96, 26.04]; ±0.05 rounding). v1 explicitly RETRACTED in STATUS. |
| 10 | Step E pairwise τ_v4 vs σ_v4 = 4-4 TIE | ✅ exact | 4-4 with 50% A-position rate (no bias) |
| 11 | Pre-realignment μ baseline = 8.88/20 (last-3-iter mean) | ✅ exact | iters 4 / 6 / 8 = 9.38, 8.00, 9.25 → mean 8.88 |
| 12 | κ-v1 smoke verdict = 24.38 | ⚠ INVALID | STATUS itself says "INVALIDATED"; raw iter-0 audit = 24.88 (not 24.38). The 24.38 number doesn't appear in raw data; could be a typo or pre-iter1 value. |
| 13 | μ-v7 OPD iter 4 = 26.625, iter 5 = 23.125, combined = 24.875 | ✅ exact | iter 4: [27, 28, 27, 31, 28, 24, 22, 26]/8 = 26.625; iter 5: [23, 18, 31, 22, 25, 22, 20, 24]/8 = 23.125 |
| 14 | μ-v7 OPD pairwise vs μ-v4 (no specific tally claimed) | data-confirmed | OPD wins 8-0 in `mu_v7_opd_full_pairwise_v_muv4` |
| 15 | Phase 6 7-baseline pairwise σ vs δ = 6-2 | ✅ exact | σ vs α and δ vs α flagged with A-pos 88% (outside ±25%, M8 caveat) |

**Bottom line**: Phase 2 baseline matrix and Phase 2 SDPO results are SOLID. Cross-goal Phase 4a and Step E retraction also SOLID. Phase 5 in-loop /20 scores would need re-verification by reading the training metrics (not the audit_responses). κ-v1 24.38 number is either a typo or refers to a different iter; in any case the broader claim is INVALIDATED and not load-bearing.


## 9. Paper-readiness verdict (per-finding)

This is the curated answer to your underlying question: "what can go in the paper, what can't?"

### Findings green-lit for paper

| Finding | What it says | Paper section | Risks |
|---|---|---|---|
| F1 — σ oracle lift | Frozen 30B + slim oracle: 15.62 → 25.25 (+9.63 / 45). Single biggest effect in baseline matrix. | Results table 1 | None substantive (need to honestly note "Opus-generated oracle is a ceiling estimate") |
| F2 — μ-v4 +2.75 over σ | Plan-level off-policy IS (D5 in-house) at lr=5e-5, iter 4: 28.00 (+2.75). Pairwise 20/24 (83.3%). | Headline result | Single-goal scope; multi-round instability past iter 4. |
| F3 — multi-round SDPO instability | All 3 lr regimes (1e-5 flat / 2e-4 collapse / 5e-5 cliff) destabilize. Replicates Hübotter §4. | Results / Discussion | Naming: do NOT call this "multi-round SDPO" without disambiguation; it's the in-house variant. |
| F4 — critique-token-blindness | PUCT/J_β content transfers to TEACHER (PICK) but NEVER to STUDENT (EVAL) at any iter, any lr. | Discussion / mechanism | Implies the +2.75 audit lift is NOT what we hypothesized. Need to be honest about this. |
| F5/F6 — audit methodology | Daemon audit saturates at anchor (F6); audit_v3 ISOLATED is the gold standard (F5). | Methods | Frame as "we developed a per-plan isolated audit because batched audit had templating bias" — fine. |
| F7 — cross-goal NULL | n=24 pairwise + strict + 2-plan all converge: μ-v4 LoRA does NOT transfer to forward-citation papers. M8 anchoring artifact retracted. | Discussion / Limitations | Negative result, strong. Don't paper over. |
| F8 v2 — τ pipeline ≈ σ | τ_v4_clean inference-time pipeline statistically comparable (CI overlaps, 4-4 TIE pairwise) to σ baseline. 3 characterized failure modes. | Results / Discussion | Frame as "we built it; it ≈ σ; here's why" — negative-with-mechanism. |
| F9 — continual SDPO 4-cell NULL | All 4 cells (anchor saturation × Adam state × anchor_ce) fail H1 forward learning. 3-protocol agreement. anchor_ce backfires. | Discussion / Limitations | Most rigorous negative finding in project. |
| M8 — audit-pairwise divergence | 2-plan/subagent batching inflates close-cluster Δ by ~2-3 points. Strict 1-plan + pairwise required for Δ < 5/45. | Methods / Methodology | Real methodology contribution. Document as the project's standing protocol. |
| F13 — μ-v7 OPD architecture | Canonical-friendly on-policy IS variant: pairwise 8/8 over μ-v4 on TTT-D, cliff delayed +2 iters. | Results / Architecture ablation | Iter-4 peak 26.625 < μ-v4 28.00 — be honest about both numbers. |
| F14 — μ-v7 OPD cross-goal NULL | On-policy variant does NOT rescue cross-goal transfer (aggregate 23.458). | Discussion | Confirms cross-goal failure is not a (c)-HER artifact. |

### Findings to KEEP OFF the paper (or limit to footnote)

| Finding / claim | Status | Why off |
|---|---|---|
| Phase 0c smoke v1/v2/v3 numbers | superseded | use Phase 2F audit_v3 ISOLATED instead |
| Phase 0.6 pre-realignment μ baseline = 8.88/20 | archived | do not cite — pre-realignment, oracle was wrong format |
| μ-v3 collapse-at-2e-4 specifics | mention only as F3 lr ablation | the failure mode (Chinese bleed, JSON leaks) is colorful but distracting |
| F8 v1 "τ_v4 = 26.50 / 7-1 STRONG" | RETRACTED | use F8 v2 (= 24.50, 4-4 TIE) instead. Footnote the retraction. |
| κ-v1 smoke "24.38" verdict | RETRACTED | "cost-saving compromise; SDPO advantage on cold-start text is not the same test as real critic" — don't cite |
| Phase 4a "Strong-mixed 2/3 pass" | RETRACTED | use F7 final NULL at n=24 |
| "SDPO" terminology without disambiguation | needs care | always say "in-house off-policy IS-loss variant" or "Hübotter SDPO with TEACHER sampling instead of STUDENT" |

### Recommended paper structure (one path)

**Title**: *D5 — Self-distillation with privileged factual observation for long-form research-plan generation*

**Sections**:
1. **Introduction**: long-form research plans as a benchmark for retrieval+SDPO. Open-source 30B can or can't reach hand-curated baseline?
2. **Related work**: SDPO (Hübotter), OPSD (Zhao), RLAD, FLARE, IRCoT, ExpeL, DR Tulu — what's distinct.
3. **Methods**:
   - 3.1 Pipeline: oracle abstraction + plan-level off-policy IS-loss + Opus critic with privileged source-paper info.
   - 3.2 Audit methodology (audit_v3 ISOLATED, 9-dim rubric, M4).
   - 3.3 Cross-validation protocols (pairwise, strict 1-plan, M8).
4. **Results**:
   - 4.1 Baseline matrix (Table 1; F1; σ +9.63 over ξ).
   - 4.2 Production μ-v4 (F2; +2.75 over σ; pairwise 20/24).
   - 4.3 SDPO multi-round instability ablation (F3; 3-lr).
   - 4.4 Phase 3 inference-time τ pipeline ≈ σ (F8 v2; 3 failure modes).
   - 4.5 Cross-goal NULL (F7; n=24).
   - 4.6 Continual SDPO NULL (F9; 4-cell grid).
   - 4.7 Architecture ablation: on-policy IS-loss (μ-v7-opd) (F13/F14; helps in-distribution, doesn't fix cross-goal).
5. **Discussion**:
   - critique-token-blindness (F4) as mechanism for F3.
   - audit-pairwise divergence (M8) as standing methodology contribution.
6. **Limitations**: single-goal scope; Opus oracle = ceiling; n=8 plans/iter; word-cap (600-750) caps T3/U5; multi-round instability not solved.
7. **Conclusion**: positive headline result + properly-characterized negatives.

### What is NOT proven enough for paper claim

- "SDPO transfers across goals" — F7 NULL; do not claim.
- "Continual SDPO works with regularization" — F9 NULL with anchor_ce backfire; do not claim.
- "Self-review can replace Opus critic" — Smoke A 8-0 against; do not claim.
- "Inference-time distillation > frozen oracle" — F8 v2 4-4 TIE; do not claim.
- "Single-paper LoRA encodes general planning skill" — F7 NULL on per-dim breakdown shows U5/T3/T4 are goal-shaped; do not claim.
- "Canonical SDPO" anywhere in the prose without clarification — D5 trainers are the in-house variant.

---

## 10. Bottom line

D5's paper-grade contributions are:
1. **F2** (μ-v4 +2.75 over σ) — single quantitative headline.
2. **F1** (σ +9.63 over ξ) — baseline-architecture finding.
3. **M8** (audit-pairwise divergence) — methodology finding worth a section in itself.
4. **F7** + **F9** + **F8 v2** — three properly-characterized negative results that make the paper honest.
5. **F4** — critique-token-blindness mechanism that explains F3.

The paper is ~95% ready content-wise. Pending items:
- (optional) Phase 3 τ-1200 ablation to characterize length-cap sensitivity.
- (optional) σ baseline K=4-8 reseed to nail population mean (Step F).
- (mandatory) prose pass to enforce "in-house off-policy IS" terminology + retract F8 v1 in any place it still appears.
- (mandatory) reconcile μ-v4 vs μ-v7-opd as architecture-ablation rather than competing headlines.

ICLR 2027 deadline ~2026-10. Plenty of time.
